# 34. コスト計算と300取引の再現
出典：FX (3).ipynb、元セルindex [80, 81, 82, 83]。保存出力は results/imported_fx3/。
研究履歴の原本です。Notebookの変数・価格CSV・学習済みファイルに依存します。
失敗した試行も保管しています。一括実行やAPI接続を開始する入口ではありません。
元コード内の指示・自動判定名は資料として保存しています。独立した検証済みの結論とは区別してください。


## 元セルindex 80
構文状態：valid


In [ ]:
# ============================================================
# FINAL 2026 PAPER EXECUTION REPLAY
# ONE CELL / STATEFUL / FROZEN CONTRACT
#
# NO TRAINING
# NO OPTIMIZATION
# NO PARAMETER CHANGE
# NO REAL ORDERS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
import json

try:
    from IPython.display import display
except Exception:
    display = print


pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 350)
pd.set_option("display.max_rows", 100)


# ============================================================
# 0. FROZEN CONTRACT
# ============================================================

EXPECTED_CHAMPION = "BASE_PLUS_REGIME"
EXPECTED_TRADES = 300
EXPECTED_YEAR = 2026

STARTING_EQUITY = 1_000_000.0

FROZEN_COST = 4e-05

ENTRY_MODE = "OPEN_AT_TIME"
EXIT_MODE = "PREVIOUS_BAR_CLOSE"

SIGNAL_TO_ENTRY_MIN = 15
HOLDING_MIN = 30

PRICE_TOL = 1e-12
RETURN_TOL = 1e-12
METRIC_TOL = 1e-10


# ============================================================
# 1. HELPERS
# ============================================================

def safe_utc_series(values):

    try:

        if isinstance(values, pd.Series):
            s = values.reset_index(drop=True)

        elif isinstance(values, pd.Index):
            s = pd.Series(values.to_numpy())

        elif isinstance(values, np.ndarray):
            s = pd.Series(values)

        elif isinstance(values, (list, tuple)):
            s = pd.Series(list(values))

        else:
            s = pd.Series([values])

        return pd.to_datetime(
            s,
            utc=True,
            errors="coerce"
        ).reset_index(drop=True)

    except Exception:

        return pd.Series(
            [],
            dtype="datetime64[ns, UTC]"
        )


def safe_numeric(values):

    try:

        return pd.to_numeric(
            pd.Series(values).reset_index(drop=True),
            errors="coerce"
        )

    except Exception:

        return pd.Series(dtype=float)


def find_col(df, candidates):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        key = str(candidate).strip().lower()

        if key in lookup:
            return lookup[key]

    return None


def max_abs(series):

    x = safe_numeric(series)

    x = (
        x.replace([np.inf, -np.inf], np.nan)
         .dropna()
    )

    if len(x) == 0:
        return np.nan

    return float(x.abs().max())


def calculate_metrics(returns, starting_equity=1.0):

    r = safe_numeric(returns)

    r = (
        r.replace([np.inf, -np.inf], np.nan)
         .dropna()
    )

    if len(r) == 0:

        return {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "profit_factor": np.nan,
            "growth": np.nan,
            "max_dd": np.nan,
            "return_to_dd": np.nan,
            "final_equity": np.nan,
        }

    positive = r[r > 0].sum()
    negative = r[r < 0].sum()

    if negative < 0:
        pf = float(positive / abs(negative))
    elif positive > 0:
        pf = np.inf
    else:
        pf = np.nan

    equity_curve = (
        float(starting_equity)
        *
        (1.0 + r).cumprod()
    )

    peak = equity_curve.cummax()

    drawdown = (
        equity_curve
        /
        peak
        -
        1.0
    )

    growth = float(
        equity_curve.iloc[-1]
        /
        float(starting_equity)
        -
        1.0
    )

    max_dd = float(
        drawdown.min()
    )

    if max_dd < 0:

        return_to_dd = float(
            growth
            /
            abs(max_dd)
        )

    else:

        return_to_dd = np.inf

    return {
        "trades": int(len(r)),
        "win_rate": float((r > 0).mean()),
        "avg_return": float(r.mean()),
        "profit_factor": pf,
        "growth": growth,
        "max_dd": max_dd,
        "return_to_dd": return_to_dd,
        "final_equity": float(equity_curve.iloc[-1]),
    }


def safe_stop(message, decision):

    print()
    print("=" * 120)
    print("STOP")
    print("=" * 120)

    print(message)

    print()
    print("Decision:", decision)

    print()
    print(
        "Champion/model/features were NOT modified."
    )

    print(
        "No paper/live order was sent."
    )

    globals()[
        "FINAL_PAPER_REPLAY_OK"
    ] = False

    globals()[
        "FINAL_PAPER_REPLAY_DECISION"
    ] = decision

    return {
        "ok": False,
        "decision": decision,
        "message": message,
    }


# ============================================================
# 2. FINAL REPLAY
# ============================================================

def run_final_2026_paper_execution_replay():

    try:

        print("=" * 120)
        print("FINAL 2026 PAPER EXECUTION REPLAY")
        print("=" * 120)

        print("Champion:", EXPECTED_CHAMPION)
        print("Expected trades:", EXPECTED_TRADES)
        print("Entry:", ENTRY_MODE)
        print("Exit:", EXIT_MODE)
        print("Holding:", f"{HOLDING_MIN} minutes")
        print("Cost:", FROZEN_COST)
        print("Starting equity:", f"{STARTING_EQUITY:,.2f}")


        # ====================================================
        # 3. PRIOR PRICE CONTRACT MUST HAVE PASSED
        # ====================================================

        prior_price_ok = globals().get(
            "EXECUTION_PRICE_SEMANTICS_OK",
            False
        )

        prior_entry_mode = globals().get(
            "EXECUTION_PRICE_ENTRY_MODE"
        )

        prior_exit_mode = globals().get(
            "EXECUTION_PRICE_EXIT_MODE"
        )


        print()
        print("=" * 120)
        print("PRIOR EXECUTION PRICE CONTRACT")
        print("=" * 120)

        print(
            "Price semantics passed:",
            prior_price_ok
        )

        print(
            "Frozen Entry:",
            prior_entry_mode
        )

        print(
            "Frozen Exit:",
            prior_exit_mode
        )


        if not bool(prior_price_ok):

            return safe_stop(
                "Execution Price Semantics がPASSしていません。",
                "STOP_PRICE_SEMANTICS_NOT_PASSED"
            )


        if str(prior_entry_mode) != ENTRY_MODE:

            return safe_stop(
                f"Entry contract mismatch: {prior_entry_mode}",
                "STOP_ENTRY_CONTRACT_MISMATCH"
            )


        if str(prior_exit_mode) != EXIT_MODE:

            return safe_stop(
                f"Exit contract mismatch: {prior_exit_mode}",
                "STOP_EXIT_CONTRACT_MISMATCH"
            )


        # ====================================================
        # 4. HISTORICAL REFERENCE TRADES
        # ====================================================

        hist_obj = globals().get(
            "HISTORICAL_REPLAY_LIVE_TRADES"
        )


        if not isinstance(hist_obj, pd.DataFrame):

            return safe_stop(
                "HISTORICAL_REPLAY_LIVE_TRADES がありません。",
                "STOP_HISTORICAL_TRADES_MISSING"
            )


        historical = hist_obj.copy()


        # ----------------------------------------------------
        # recover signal time before reset_index
        # ----------------------------------------------------

        if "signal_time" in historical.columns:

            signal_time = safe_utc_series(
                historical["signal_time"]
            )

        elif "_diag_signal_time" in historical.columns:

            signal_time = safe_utc_series(
                historical["_diag_signal_time"]
            )

        elif isinstance(
            historical.index,
            pd.DatetimeIndex
        ):

            signal_time = safe_utc_series(
                historical.index
            )

        else:

            return safe_stop(
                "Historical signal_time を復元できません。",
                "STOP_SIGNAL_TIME_MISSING"
            )


        historical = historical.reset_index(
            drop=True
        )


        if len(historical) != EXPECTED_TRADES:

            return safe_stop(
                (
                    f"Historical trades = {len(historical)}, "
                    f"expected = {EXPECTED_TRADES}"
                ),
                "STOP_HISTORICAL_TRADE_COUNT"
            )


        # ====================================================
        # 5. RESOLVE HISTORICAL COLUMNS
        # ====================================================

        entry_col = find_col(
            historical,
            ["entry_time"]
        )

        exit_col = find_col(
            historical,
            ["exit_time"]
        )

        side_col = find_col(
            historical,
            ["side"]
        )

        size_col = find_col(
            historical,
            ["position_size"]
        )

        gross_col = find_col(
            historical,
            ["gross_return"]
        )

        net_col = find_col(
            historical,
            ["net_return"]
        )

        confidence_col = find_col(
            historical,
            ["confidence"]
        )

        p_up_col = find_col(
            historical,
            ["p_up"]
        )


        required = {
            "entry_time": entry_col,
            "exit_time": exit_col,
            "side": side_col,
            "position_size": size_col,
            "gross_return": gross_col,
            "net_return": net_col,
        }


        missing = [
            k
            for k, v in required.items()
            if v is None
        ]


        if missing:

            return safe_stop(
                f"Historical required columns missing: {missing}",
                "STOP_HISTORICAL_COLUMNS_MISSING"
            )


        reference = pd.DataFrame()

        reference["signal_time"] = (
            signal_time.to_numpy()
        )

        reference["entry_time"] = (
            safe_utc_series(
                historical[entry_col]
            ).to_numpy()
        )

        reference["exit_time"] = (
            safe_utc_series(
                historical[exit_col]
            ).to_numpy()
        )

        reference["side"] = (
            historical[side_col]
            .astype(str)
            .str.strip()
            .str.upper()
            .reset_index(drop=True)
        )

        reference["position_size"] = (
            safe_numeric(
                historical[size_col]
            ).to_numpy()
        )

        reference["stored_gross_return"] = (
            safe_numeric(
                historical[gross_col]
            ).to_numpy()
        )

        reference["stored_net_return"] = (
            safe_numeric(
                historical[net_col]
            ).to_numpy()
        )


        if confidence_col is not None:

            reference["confidence"] = (
                safe_numeric(
                    historical[confidence_col]
                ).to_numpy()
            )

        else:

            reference["confidence"] = np.nan


        if p_up_col is not None:

            reference["p_up"] = (
                safe_numeric(
                    historical[p_up_col]
                ).to_numpy()
            )

        else:

            reference["p_up"] = np.nan


        # ----------------------------------------------------
        # validate complete required rows
        # ----------------------------------------------------

        required_check_cols = [
            "signal_time",
            "entry_time",
            "exit_time",
            "side",
            "position_size",
            "stored_gross_return",
            "stored_net_return",
        ]


        if (
            reference[required_check_cols]
            .isna()
            .any()
            .any()
        ):

            return safe_stop(
                "Historical reference に欠損があります。",
                "STOP_HISTORICAL_REFERENCE_NAN"
            )


        if not reference[
            "side"
        ].isin(
            ["BUY", "SELL"]
        ).all():

            return safe_stop(
                "Historical side に BUY/SELL 以外があります。",
                "STOP_INVALID_SIDE"
            )


        reference = (
            reference
            .sort_values("signal_time")
            .reset_index(drop=True)
        )


        reference["trade_id"] = np.arange(
            len(reference),
            dtype=int
        )


        print()
        print("=" * 120)
        print("HISTORICAL REFERENCE")
        print("=" * 120)

        print("Rows:", len(reference))

        print(
            "First signal:",
            reference["signal_time"].min()
        )

        print(
            "Last exit:",
            reference["exit_time"].max()
        )


        # ====================================================
        # 6. TIMING CONTRACT RECHECK
        # ====================================================

        signal_to_entry = (

            (
                reference["entry_time"]
                -
                reference["signal_time"]
            )
            .dt.total_seconds()
            /
            60.0
        )


        holding = (

            (
                reference["exit_time"]
                -
                reference["entry_time"]
            )
            .dt.total_seconds()
            /
            60.0
        )


        timing_signal_ok = bool(
            np.isclose(
                signal_to_entry,
                SIGNAL_TO_ENTRY_MIN
            ).all()
        )


        timing_hold_ok = bool(
            np.isclose(
                holding,
                HOLDING_MIN
            ).all()
        )


        ordered_ref = (
            reference
            .sort_values("entry_time")
            .reset_index(drop=True)
        )


        overlap_reference = int(

            (
                ordered_ref["entry_time"]
                <
                ordered_ref["exit_time"].shift(1)
            )
            .fillna(False)
            .sum()
        )


        duplicate_signals = int(
            reference[
                "signal_time"
            ].duplicated().sum()
        )


        print()
        print("=" * 120)
        print("TIMING / INTEGRITY")
        print("=" * 120)

        print(
            "Signal -> Entry = 15m:",
            timing_signal_ok
        )

        print(
            "Entry -> Exit = 30m:",
            timing_hold_ok
        )

        print(
            "Duplicate signals:",
            duplicate_signals
        )

        print(
            "Historical overlap:",
            overlap_reference
        )


        if not (
            timing_signal_ok
            and timing_hold_ok
            and duplicate_signals == 0
            and overlap_reference == 0
        ):

            return safe_stop(
                "Historical timing/integrity contract mismatch.",
                "STOP_TIMING_INTEGRITY"
            )


        # ====================================================
        # 7. CANONICAL HISTORY
        # ====================================================

        canonical_obj = globals().get(
            "PRODUCTION_CANONICAL_HISTORY"
        )


        if not isinstance(
            canonical_obj,
            pd.DataFrame
        ):

            return safe_stop(
                "PRODUCTION_CANONICAL_HISTORY がありません。",
                "STOP_CANONICAL_HISTORY_MISSING"
            )


        raw_bars = canonical_obj.copy()


        if len(raw_bars) != 265_905:

            return safe_stop(
                (
                    "Canonical History row count changed: "
                    f"{len(raw_bars):,}"
                ),
                "STOP_CANONICAL_ROW_COUNT_CHANGED"
            )


        # timestamp
        if isinstance(
            raw_bars.index,
            pd.DatetimeIndex
        ):

            timestamps = safe_utc_series(
                raw_bars.index
            )

        else:

            time_col = find_col(
                raw_bars,
                [
                    "timestamp",
                    "datetime",
                    "time"
                ]
            )

            if time_col is None:

                return safe_stop(
                    "Canonical timestamp not found.",
                    "STOP_CANONICAL_TIMESTAMP_MISSING"
                )

            timestamps = safe_utc_series(
                raw_bars[time_col]
            )


        open_col = find_col(
            raw_bars,
            ["open"]
        )

        high_col = find_col(
            raw_bars,
            ["high"]
        )

        low_col = find_col(
            raw_bars,
            ["low"]
        )

        close_col = find_col(
            raw_bars,
            ["close"]
        )


        if any(
            c is None
            for c in [
                open_col,
                high_col,
                low_col,
                close_col
            ]
        ):

            return safe_stop(
                "Canonical OHLC columns missing.",
                "STOP_CANONICAL_OHLC_MISSING"
            )


        bars = pd.DataFrame({
            "timestamp": timestamps,
            "open": safe_numeric(
                raw_bars[open_col]
            ),
            "high": safe_numeric(
                raw_bars[high_col]
            ),
            "low": safe_numeric(
                raw_bars[low_col]
            ),
            "close": safe_numeric(
                raw_bars[close_col]
            ),
        })


        bars = (
            bars
            .dropna()
            .sort_values("timestamp")
            .drop_duplicates(
                "timestamp",
                keep="last"
            )
            .set_index("timestamp")
        )


        if len(bars) != 265_905:

            return safe_stop(
                (
                    "Canonical normalization changed row count: "
                    f"{len(bars):,}"
                ),
                "STOP_CANONICAL_NORMALIZATION_CHANGED"
            )


        print()
        print("=" * 120)
        print("CANONICAL DATA")
        print("=" * 120)

        print(
            "Rows:",
            f"{len(bars):,}"
        )

        print(
            "Period:",
            bars.index.min(),
            "->",
            bars.index.max()
        )


        # ====================================================
        # 8. ENSURE ALL EXECUTION PRICES EXIST
        # ====================================================

        entry_lookup = pd.DatetimeIndex(
            reference["entry_time"]
        )

        exit_lookup_previous = pd.DatetimeIndex(
            reference["exit_time"]
        ) - pd.Timedelta(minutes=15)


        missing_entry_prices = int(
            bars["open"]
            .reindex(entry_lookup)
            .isna()
            .sum()
        )


        missing_exit_prices = int(
            bars["close"]
            .reindex(exit_lookup_previous)
            .isna()
            .sum()
        )


        print()
        print("=" * 120)
        print("EXECUTION PRICE AVAILABILITY")
        print("=" * 120)

        print(
            "Missing entry prices:",
            missing_entry_prices
        )

        print(
            "Missing exit prices:",
            missing_exit_prices
        )


        if (
            missing_entry_prices > 0
            or
            missing_exit_prices > 0
        ):

            return safe_stop(
                "Some frozen execution prices are missing.",
                "STOP_EXECUTION_PRICES_MISSING"
            )


        # ====================================================
        # 9. VERIFY FROZEN COST FROM HISTORICAL ACCOUNTING
        # ====================================================

        implied_cost = (

            reference[
                "stored_gross_return"
            ]

            *

            reference[
                "position_size"
            ]

            -

            reference[
                "stored_net_return"
            ]
        )


        implied_cost_median = float(
            implied_cost.median()
        )

        implied_cost_max_dev = float(

            (
                implied_cost
                -
                FROZEN_COST
            )
            .abs()
            .max()
        )


        cost_contract_ok = bool(
            implied_cost_max_dev
            <=
            RETURN_TOL
        )


        print()
        print("=" * 120)
        print("FROZEN COST ACCOUNTING")
        print("=" * 120)

        print(
            "Frozen cost:",
            FROZEN_COST
        )

        print(
            "Median implied cost:",
            implied_cost_median
        )

        print(
            "Max cost difference:",
            implied_cost_max_dev
        )

        print(
            "Cost contract passed:",
            cost_contract_ok
        )


        if not cost_contract_ok:

            return safe_stop(
                "Historical net-return accounting does not match 4e-05 cost.",
                "STOP_COST_CONTRACT_MISMATCH"
            )


        # ====================================================
        # 10. STATEFUL PAPER EXECUTION ENGINE
        # ====================================================

        class FrozenPaperReplayEngine:

            def __init__(
                self,
                canonical,
                starting_equity,
                cost
            ):

                self.bars = canonical

                self.starting_equity = float(
                    starting_equity
                )

                self.equity = float(
                    starting_equity
                )

                self.cost = float(cost)

                self.pending_order = None
                self.position = None

                self.executed = []
                self.events = []
                self.failures = []


            def log_event(
                self,
                timestamp,
                event,
                trade_id=None,
                detail=None
            ):

                self.events.append({
                    "timestamp": timestamp,
                    "event": event,
                    "trade_id": trade_id,
                    "detail": detail,
                })


            def fail(
                self,
                timestamp,
                code,
                trade_id=None,
                detail=None
            ):

                self.failures.append({
                    "timestamp": timestamp,
                    "failure": code,
                    "trade_id": trade_id,
                    "detail": detail,
                })


            def submit_signal(
                self,
                row
            ):

                ts = row["signal_time"]

                if self.pending_order is not None:

                    self.fail(
                        ts,
                        "PENDING_ORDER_ALREADY_EXISTS",
                        row["trade_id"]
                    )

                    return


                # Existing position is allowed only if its
                # scheduled exit is exactly at this signal
                # boundary or later.
                # No new entry occurs until +15m.
                self.pending_order = {
                    "trade_id":
                        int(row["trade_id"]),

                    "signal_time":
                        row["signal_time"],

                    "entry_time":
                        row["entry_time"],

                    "exit_time":
                        row["exit_time"],

                    "side":
                        row["side"],

                    "position_size":
                        float(row["position_size"]),

                    "confidence":
                        row["confidence"],

                    "p_up":
                        row["p_up"],
                }

                self.log_event(
                    ts,
                    "SIGNAL_ACCEPTED",
                    row["trade_id"],
                    row["side"]
                )


            def enter_if_due(
                self,
                ts
            ):

                if self.pending_order is None:
                    return


                if (
                    self.pending_order[
                        "entry_time"
                    ]
                    != ts
                ):
                    return


                if self.position is not None:

                    self.fail(
                        ts,
                        "OVERLAP_ENTRY",
                        self.pending_order[
                            "trade_id"
                        ]
                    )

                    return


                if ts not in self.bars.index:

                    self.fail(
                        ts,
                        "ENTRY_BAR_MISSING",
                        self.pending_order[
                            "trade_id"
                        ]
                    )

                    return


                entry_price = float(
                    self.bars.at[
                        ts,
                        "open"
                    ]
                )


                self.position = {
                    **self.pending_order,
                    "entry_price":
                        entry_price,
                    "equity_before":
                        float(self.equity),
                }


                self.log_event(
                    ts,
                    "ENTRY",
                    self.position[
                        "trade_id"
                    ],
                    entry_price
                )


                self.pending_order = None


            def exit_if_due(
                self,
                ts
            ):

                if self.position is None:
                    return


                if (
                    self.position[
                        "exit_time"
                    ]
                    != ts
                ):
                    return


                previous_bar_time = (
                    ts
                    -
                    pd.Timedelta(
                        minutes=15
                    )
                )


                if (
                    previous_bar_time
                    not in self.bars.index
                ):

                    self.fail(
                        ts,
                        "EXIT_PRICE_BAR_MISSING",
                        self.position[
                            "trade_id"
                        ]
                    )

                    return


                exit_price = float(
                    self.bars.at[
                        previous_bar_time,
                        "close"
                    ]
                )


                entry_price = float(
                    self.position[
                        "entry_price"
                    ]
                )


                side = self.position[
                    "side"
                ]


                raw_return = (
                    exit_price
                    /
                    entry_price
                    -
                    1.0
                )


                if side == "BUY":

                    gross_return = float(
                        raw_return
                    )

                elif side == "SELL":

                    gross_return = float(
                        -raw_return
                    )

                else:

                    self.fail(
                        ts,
                        "INVALID_POSITION_SIDE",
                        self.position[
                            "trade_id"
                        ]
                    )

                    return


                position_size = float(
                    self.position[
                        "position_size"
                    ]
                )


                net_return = float(

                    gross_return
                    *
                    position_size
                    -
                    self.cost
                )


                equity_before = float(
                    self.equity
                )


                equity_after = float(

                    equity_before
                    *
                    (
                        1.0
                        +
                        net_return
                    )
                )


                self.equity = equity_after


                completed = {
                    **self.position,

                    "exit_price":
                        exit_price,

                    "gross_return":
                        gross_return,

                    "net_return":
                        net_return,

                    "equity_before":
                        equity_before,

                    "equity_after":
                        equity_after,
                }


                self.executed.append(
                    completed
                )


                self.log_event(
                    ts,
                    "EXIT",
                    completed[
                        "trade_id"
                    ],
                    exit_price
                )


                self.position = None


        # ====================================================
        # 11. BUILD EVENT MAPS
        # ====================================================

        signal_map = {}

        for _, row in reference.iterrows():

            ts = row["signal_time"]

            signal_map.setdefault(
                ts,
                []
            ).append(
                row
            )


        # ====================================================
        # 12. REPLAY ALL CANONICAL 15m BOUNDARIES
        # ====================================================

        engine = FrozenPaperReplayEngine(
            canonical=bars,
            starting_equity=STARTING_EQUITY,
            cost=FROZEN_COST
        )


        first_event_time = reference[
            "signal_time"
        ].min()


        last_event_time = reference[
            "exit_time"
        ].max()


        replay_boundaries = bars.index[
            (
                bars.index
                >=
                first_event_time
            )
            &
            (
                bars.index
                <=
                last_event_time
            )
        ]


        print()
        print("=" * 120)
        print("RUNNING STATEFUL PAPER REPLAY")
        print("=" * 120)

        print(
            "15m boundaries:",
            f"{len(replay_boundaries):,}"
        )


        for ts in replay_boundaries:

            # -----------------------------------------------
            # 1. EXIT old position first
            #    allows back-to-back positions at boundary
            # -----------------------------------------------

            engine.exit_if_due(ts)


            # -----------------------------------------------
            # 2. ENTER pending order
            # -----------------------------------------------

            engine.enter_if_due(ts)


            # -----------------------------------------------
            # 3. PROCESS new signal
            # -----------------------------------------------

            if ts in signal_map:

                for row in signal_map[ts]:

                    engine.submit_signal(
                        row
                    )


        # One final exit check is normally unnecessary,
        # but harmless because last exit boundary is included.


        replay = pd.DataFrame(
            engine.executed
        )


        failures = pd.DataFrame(
            engine.failures
        )


        events = pd.DataFrame(
            engine.events
        )


        print()
        print(
            "Replay loop complete."
        )

        print(
            "Executed:",
            len(replay)
        )

        print(
            "Failures:",
            len(failures)
        )

        print(
            "Pending order remaining:",
            engine.pending_order is not None
        )

        print(
            "Open position remaining:",
            engine.position is not None
        )


        # ====================================================
        # 13. ENGINE STATE VALIDATION
        # ====================================================

        no_failures = (
            len(failures) == 0
        )

        no_pending = (
            engine.pending_order is None
        )

        no_position = (
            engine.position is None
        )

        executed_300 = (
            len(replay)
            ==
            EXPECTED_TRADES
        )


        if not (
            no_failures
            and no_pending
            and no_position
            and executed_300
        ):

            print()

            if len(failures):

                print("=" * 120)
                print("ENGINE FAILURES")
                print("=" * 120)

                display(
                    failures.head(30)
                )

            return safe_stop(
                "Stateful Paper Replay mechanics did not complete cleanly.",
                "STOP_STATEFUL_REPLAY_FAILURE"
            )


        # ====================================================
        # 14. ALIGN REPLAY WITH HISTORICAL REFERENCE
        # ====================================================

        replay = (
            replay
            .sort_values(
                "trade_id"
            )
            .reset_index(drop=True)
        )


        reference_cmp = (
            reference
            .sort_values(
                "trade_id"
            )
            .reset_index(drop=True)
        )


        # expected prices using frozen contract
        expected_entry_prices = (

            bars["open"]

            .reindex(
                pd.DatetimeIndex(
                    reference_cmp[
                        "entry_time"
                    ]
                )
            )

            .to_numpy()
        )


        expected_exit_times = (

            pd.DatetimeIndex(
                reference_cmp[
                    "exit_time"
                ]
            )

            -

            pd.Timedelta(
                minutes=15
            )
        )


        expected_exit_prices = (

            bars["close"]

            .reindex(
                expected_exit_times
            )

            .to_numpy()
        )


        comparison = pd.DataFrame({

            "trade_id":
                reference_cmp[
                    "trade_id"
                ],

            "signal_time_expected":
                reference_cmp[
                    "signal_time"
                ],

            "signal_time_replay":
                replay[
                    "signal_time"
                ],

            "entry_time_expected":
                reference_cmp[
                    "entry_time"
                ],

            "entry_time_replay":
                replay[
                    "entry_time"
                ],

            "exit_time_expected":
                reference_cmp[
                    "exit_time"
                ],

            "exit_time_replay":
                replay[
                    "exit_time"
                ],

            "side_expected":
                reference_cmp[
                    "side"
                ],

            "side_replay":
                replay[
                    "side"
                ],

            "size_expected":
                reference_cmp[
                    "position_size"
                ],

            "size_replay":
                replay[
                    "position_size"
                ],

            "entry_price_expected":
                expected_entry_prices,

            "entry_price_replay":
                replay[
                    "entry_price"
                ],

            "exit_price_expected":
                expected_exit_prices,

            "exit_price_replay":
                replay[
                    "exit_price"
                ],

            "gross_expected":
                reference_cmp[
                    "stored_gross_return"
                ],

            "gross_replay":
                replay[
                    "gross_return"
                ],

            "net_expected":
                reference_cmp[
                    "stored_net_return"
                ],

            "net_replay":
                replay[
                    "net_return"
                ],
        })


        # ====================================================
        # 15. EXACT PARITY CHECKS
        # ====================================================

        signal_parity = bool(
            (
                comparison[
                    "signal_time_expected"
                ]
                ==
                comparison[
                    "signal_time_replay"
                ]
            ).all()
        )


        entry_time_parity = bool(
            (
                comparison[
                    "entry_time_expected"
                ]
                ==
                comparison[
                    "entry_time_replay"
                ]
            ).all()
        )


        exit_time_parity = bool(
            (
                comparison[
                    "exit_time_expected"
                ]
                ==
                comparison[
                    "exit_time_replay"
                ]
            ).all()
        )


        side_parity = bool(
            (
                comparison[
                    "side_expected"
                ]
                ==
                comparison[
                    "side_replay"
                ]
            ).all()
        )


        max_size_diff = max_abs(
            comparison[
                "size_replay"
            ]
            -
            comparison[
                "size_expected"
            ]
        )


        max_entry_price_diff = max_abs(
            comparison[
                "entry_price_replay"
            ]
            -
            comparison[
                "entry_price_expected"
            ]
        )


        max_exit_price_diff = max_abs(
            comparison[
                "exit_price_replay"
            ]
            -
            comparison[
                "exit_price_expected"
            ]
        )


        max_gross_diff = max_abs(
            comparison[
                "gross_replay"
            ]
            -
            comparison[
                "gross_expected"
            ]
        )


        max_net_diff = max_abs(
            comparison[
                "net_replay"
            ]
            -
            comparison[
                "net_expected"
            ]
        )


        size_parity = bool(
            max_size_diff
            <=
            RETURN_TOL
        )


        entry_price_parity = bool(
            max_entry_price_diff
            <=
            PRICE_TOL
        )


        exit_price_parity = bool(
            max_exit_price_diff
            <=
            PRICE_TOL
        )


        gross_parity = bool(
            max_gross_diff
            <=
            RETURN_TOL
        )


        net_parity = bool(
            max_net_diff
            <=
            RETURN_TOL
        )


        # ====================================================
        # 16. DUPLICATE / OVERLAP / MISSING
        # ====================================================

        replay_duplicate_signals = int(
            replay[
                "signal_time"
            ].duplicated().sum()
        )


        replay_sorted = (
            replay
            .sort_values("entry_time")
            .reset_index(drop=True)
        )


        replay_overlap = int(

            (
                replay_sorted[
                    "entry_time"
                ]

                <

                replay_sorted[
                    "exit_time"
                ].shift(1)
            )
            .fillna(False)
            .sum()
        )


        reference_ids = set(
            reference_cmp[
                "trade_id"
            ].astype(int)
        )


        replay_ids = set(
            replay[
                "trade_id"
            ].astype(int)
        )


        missing_ids = sorted(
            reference_ids
            -
            replay_ids
        )


        unexpected_ids = sorted(
            replay_ids
            -
            reference_ids
        )


        # ====================================================
        # 17. PERFORMANCE PARITY
        # ====================================================

        expected_metrics = calculate_metrics(
            reference_cmp[
                "stored_net_return"
            ],
            starting_equity=STARTING_EQUITY
        )


        replay_metrics = calculate_metrics(
            replay[
                "net_return"
            ],
            starting_equity=STARTING_EQUITY
        )


        metric_rows = []

        for key in [
            "trades",
            "win_rate",
            "avg_return",
            "profit_factor",
            "growth",
            "max_dd",
            "return_to_dd",
            "final_equity",
        ]:

            expected_value = (
                expected_metrics[key]
            )

            replay_value = (
                replay_metrics[key]
            )


            if (
                isinstance(
                    expected_value,
                    (int, np.integer)
                )
                and
                isinstance(
                    replay_value,
                    (int, np.integer)
                )
            ):

                diff = abs(
                    int(expected_value)
                    -
                    int(replay_value)
                )

            elif (
                np.isinf(expected_value)
                and
                np.isinf(replay_value)
            ):

                diff = 0.0

            else:

                diff = abs(
                    float(expected_value)
                    -
                    float(replay_value)
                )


            metric_rows.append({
                "metric": key,
                "expected": expected_value,
                "replay": replay_value,
                "abs_diff": diff,
            })


        metric_compare = pd.DataFrame(
            metric_rows
        )


        performance_parity = bool(

            (
                metric_compare[
                    "abs_diff"
                ]
                <=
                METRIC_TOL
            )
            .all()
        )


        # ====================================================
        # 18. EQUITY PATH PARITY
        # ====================================================

        expected_equity = (

            STARTING_EQUITY

            *

            (
                1.0
                +
                reference_cmp[
                    "stored_net_return"
                ]
            )
            .cumprod()
        )


        replay_equity = (
            replay[
                "equity_after"
            ]
            .reset_index(drop=True)
        )


        equity_diff = (

            replay_equity
            -
            expected_equity.reset_index(
                drop=True
            )
        )


        max_equity_diff = max_abs(
            equity_diff
        )


        # use relative tolerance because equity ≈ 1e6
        equity_parity = bool(
            max_equity_diff
            <=
            1e-6
        )


        # ====================================================
        # 19. FINAL CHECK TABLE
        # ====================================================

        checks = {
            "historical_trades_300":
                len(reference_cmp)
                ==
                EXPECTED_TRADES,

            "replay_trades_300":
                len(replay)
                ==
                EXPECTED_TRADES,

            "missing_trades_0":
                len(missing_ids)
                ==
                0,

            "unexpected_trades_0":
                len(unexpected_ids)
                ==
                0,

            "duplicate_signals_0":
                replay_duplicate_signals
                ==
                0,

            "overlap_0":
                replay_overlap
                ==
                0,

            "signal_timestamp_parity":
                signal_parity,

            "entry_timestamp_parity":
                entry_time_parity,

            "exit_timestamp_parity":
                exit_time_parity,

            "side_parity":
                side_parity,

            "position_size_parity":
                size_parity,

            "entry_price_parity":
                entry_price_parity,

            "exit_price_parity":
                exit_price_parity,

            "gross_return_parity":
                gross_parity,

            "net_return_parity":
                net_parity,

            "equity_path_parity":
                equity_parity,

            "performance_parity":
                performance_parity,

            "engine_no_failure":
                no_failures,

            "engine_flat_end":
                no_position,

            "engine_no_pending_end":
                no_pending,
        }


        checks_df = pd.DataFrame({
            "check":
                list(checks.keys()),

            "passed":
                list(checks.values())
        })


        FINAL_PASS = bool(
            all(
                bool(v)
                for v in checks.values()
            )
        )


        # ====================================================
        # 20. REPORT
        # ====================================================

        print()
        print("=" * 120)
        print("TRADE-LEVEL PARITY")
        print("=" * 120)

        print(
            "Expected trades:",
            EXPECTED_TRADES
        )

        print(
            "Replay trades:",
            len(replay)
        )

        print(
            "Missing:",
            len(missing_ids)
        )

        print(
            "Unexpected:",
            len(unexpected_ids)
        )

        print(
            "Duplicate signals:",
            replay_duplicate_signals
        )

        print(
            "Overlap:",
            replay_overlap
        )


        print()
        print("=" * 120)
        print("MAXIMUM DIFFERENCES")
        print("=" * 120)

        print(
            "Position size:",
            max_size_diff
        )

        print(
            "Entry price:",
            max_entry_price_diff
        )

        print(
            "Exit price:",
            max_exit_price_diff
        )

        print(
            "Gross return:",
            max_gross_diff
        )

        print(
            "Net return:",
            max_net_diff
        )

        print(
            "Equity:",
            max_equity_diff
        )


        print()
        print("=" * 120)
        print("EXPECTED PERFORMANCE")
        print("=" * 120)

        for k, v in expected_metrics.items():
            print(
                f"{k}: {v}"
            )


        print()
        print("=" * 120)
        print("REPLAY PERFORMANCE")
        print("=" * 120)

        for k, v in replay_metrics.items():
            print(
                f"{k}: {v}"
            )


        print()
        print("=" * 120)
        print("PERFORMANCE COMPARISON")
        print("=" * 120)

        display(
            metric_compare
        )


        print()
        print("=" * 120)
        print("FINAL CHECKS")
        print("=" * 120)

        display(
            checks_df
        )


        # ====================================================
        # 21. SHOW WORST TRADE DIFFERENCES
        # ====================================================

        comparison[
            "entry_price_diff"
        ] = (

            comparison[
                "entry_price_replay"
            ]

            -

            comparison[
                "entry_price_expected"
            ]
        )


        comparison[
            "exit_price_diff"
        ] = (

            comparison[
                "exit_price_replay"
            ]

            -

            comparison[
                "exit_price_expected"
            ]
        )


        comparison[
            "gross_diff"
        ] = (

            comparison[
                "gross_replay"
            ]

            -

            comparison[
                "gross_expected"
            ]
        )


        comparison[
            "net_diff"
        ] = (

            comparison[
                "net_replay"
            ]

            -

            comparison[
                "net_expected"
            ]
        )


        comparison[
            "max_trade_abs_diff"
        ] = (

            comparison[
                [
                    "entry_price_diff",
                    "exit_price_diff",
                    "gross_diff",
                    "net_diff",
                ]
            ]
            .abs()
            .max(axis=1)
        )


        print()
        print("=" * 120)
        print("WORST 10 TRADE PARITY DIFFERENCES")
        print("=" * 120)

        display(

            comparison
            .sort_values(
                "max_trade_abs_diff",
                ascending=False
            )
            .head(10)
        )


        # ====================================================
        # 22. SAVE NOTEBOOK VARIABLES
        # ====================================================

        globals()[
            "PRODUCTION_EXECUTION_REPLAY_2026"
        ] = replay.copy()


        globals()[
            "PRODUCTION_EXECUTION_REPLAY_EVENTS"
        ] = events.copy()


        globals()[
            "PRODUCTION_EXECUTION_REPLAY_FAILURES"
        ] = failures.copy()


        globals()[
            "PRODUCTION_EXECUTION_REPLAY_COMPARISON"
        ] = comparison.copy()


        globals()[
            "PRODUCTION_EXECUTION_REPLAY_METRICS"
        ] = metric_compare.copy()


        globals()[
            "PRODUCTION_EXECUTION_REPLAY_CHECKS"
        ] = checks_df.copy()


        globals()[
            "FINAL_PAPER_REPLAY_OK"
        ] = FINAL_PASS


        # ====================================================
        # 23. OPTIONAL SAVE
        # ====================================================

        saved_directory = None


        live_dir = globals().get(
            "LIVE_CHAMPION_DIR"
        )


        if live_dir is not None:

            try:

                saved_directory = (

                    Path(live_dir)

                    /

                    "runtime"

                    /

                    "paper"

                    /

                    "final_replay_2026"
                )


                saved_directory.mkdir(
                    parents=True,
                    exist_ok=True
                )


                replay.to_csv(
                    saved_directory
                    /
                    "paper_execution_replay_2026.csv",
                    index=False
                )


                events.to_csv(
                    saved_directory
                    /
                    "paper_execution_events_2026.csv",
                    index=False
                )


                comparison.to_csv(
                    saved_directory
                    /
                    "paper_execution_parity_2026.csv",
                    index=False
                )


                metric_compare.to_csv(
                    saved_directory
                    /
                    "paper_execution_metrics_2026.csv",
                    index=False
                )


                checks_df.to_csv(
                    saved_directory
                    /
                    "paper_execution_checks_2026.csv",
                    index=False
                )


                report_json = {

                    "champion":
                        EXPECTED_CHAMPION,

                    "year":
                        EXPECTED_YEAR,

                    "entry_mode":
                        ENTRY_MODE,

                    "exit_mode":
                        EXIT_MODE,

                    "holding_minutes":
                        HOLDING_MIN,

                    "cost":
                        FROZEN_COST,

                    "starting_equity":
                        STARTING_EQUITY,

                    "expected_trades":
                        EXPECTED_TRADES,

                    "executed_trades":
                        int(len(replay)),

                    "missing_trades":
                        int(len(missing_ids)),

                    "unexpected_trades":
                        int(len(unexpected_ids)),

                    "duplicate_signals":
                        int(replay_duplicate_signals),

                    "overlapping_positions":
                        int(replay_overlap),

                    "max_entry_price_diff":
                        float(max_entry_price_diff),

                    "max_exit_price_diff":
                        float(max_exit_price_diff),

                    "max_gross_return_diff":
                        float(max_gross_diff),

                    "max_net_return_diff":
                        float(max_net_diff),

                    "max_equity_diff":
                        float(max_equity_diff),

                    "expected_metrics":
                        {
                            k: (
                                float(v)
                                if not isinstance(
                                    v,
                                    (int, np.integer)
                                )
                                else int(v)
                            )
                            for k, v
                            in expected_metrics.items()
                        },

                    "replay_metrics":
                        {
                            k: (
                                float(v)
                                if not isinstance(
                                    v,
                                    (int, np.integer)
                                )
                                else int(v)
                            )
                            for k, v
                            in replay_metrics.items()
                        },

                    "passed":
                        bool(FINAL_PASS),
                }


                with open(
                    saved_directory
                    /
                    "final_paper_replay_report.json",
                    "w",
                    encoding="utf-8"
                ) as f:

                    json.dump(
                        report_json,
                        f,
                        ensure_ascii=False,
                        indent=2
                    )


            except Exception as save_error:

                print()
                print(
                    "WARNING: Replay completed but "
                    "saving files failed."
                )

                print(
                    type(save_error).__name__,
                    ":",
                    str(save_error)
                )


        # ====================================================
        # 24. FINAL DECISION
        # ====================================================

        print()
        print("=" * 120)
        print("FINAL 2026 PAPER EXECUTION REPLAY DECISION")
        print("=" * 120)


        if FINAL_PASS:

            decision = (
                "PASS_FINAL_2026_PAPER_EXECUTION_REPLAY"
            )

            print(
                "FINAL PAPER EXECUTION REPLAY PASSED: True"
            )

            print()
            print(
                "FINAL DECISION:",
                decision
            )

            print()
            print(
                "300 / 300 historical trades reproduced."
            )

            print(
                "Entry / Exit / Gross / Net / Equity "
                "reproduction passed."
            )

            print(
                "No duplicate / overlap / missing / "
                "unexpected trades."
            )

            print()
            print(
                "NEXT STEP:"
            )

            print(
                "BUILD FORWARD PAPER TRADING RUNNER "
                "FOR NEW UNSEEN 15-MINUTE BARS."
            )

        else:

            decision = (
                "FAIL_DO_NOT_START_FORWARD_PAPER_TRADING"
            )

            print(
                "FINAL PAPER EXECUTION REPLAY PASSED: False"
            )

            print()
            print(
                "FINAL DECISION:",
                decision
            )

            print()
            print(
                "Do NOT start Forward Paper Trading."
            )

            print(
                "Inspect failed checks above."
            )


        globals()[
            "FINAL_PAPER_REPLAY_DECISION"
        ] = decision


        return {
            "ok":
                FINAL_PASS,

            "decision":
                decision,

            "expected_trades":
                EXPECTED_TRADES,

            "executed_trades":
                int(len(replay)),

            "missing":
                int(len(missing_ids)),

            "unexpected":
                int(len(unexpected_ids)),

            "duplicate":
                int(replay_duplicate_signals),

            "overlap":
                int(replay_overlap),

            "max_entry_price_diff":
                float(max_entry_price_diff),

            "max_exit_price_diff":
                float(max_exit_price_diff),

            "max_gross_diff":
                float(max_gross_diff),

            "max_net_diff":
                float(max_net_diff),

            "max_equity_diff":
                float(max_equity_diff),

            "saved_directory":
                (
                    str(saved_directory)
                    if saved_directory is not None
                    else None
                ),
        }


    except Exception as exc:

        # ====================================================
        # GLOBAL SAFETY CATCH
        # ====================================================

        print()
        print("=" * 120)
        print("REPLAY STOPPED SAFELY")
        print("=" * 120)

        print(
            "Exception type:",
            type(exc).__name__
        )

        print(
            "Message:",
            str(exc)
        )

        print()
        print(
            "No Champion/model/feature was modified."
        )

        print(
            "No real order was generated."
        )


        decision = (
            "STOP_FINAL_REPLAY_IMPLEMENTATION_ERROR"
        )


        globals()[
            "FINAL_PAPER_REPLAY_OK"
        ] = False


        globals()[
            "FINAL_PAPER_REPLAY_DECISION"
        ] = decision


        return {
            "ok": False,
            "decision": decision,
            "exception_type":
                type(exc).__name__,
            "exception_message":
                str(exc),
        }


# ============================================================
# 25. RUN ONCE
# ============================================================

FINAL_2026_PAPER_REPLAY_REPORT = (
    run_final_2026_paper_execution_replay()
)


print()
print("=" * 120)
print("CELL COMPLETE")
print("=" * 120)

print(
    "Decision:",
    FINAL_2026_PAPER_REPLAY_REPORT.get(
        "decision"
    )
)

print(
    "Passed:",
    FINAL_2026_PAPER_REPLAY_REPORT.get(
        "ok"
    )
)

# ============================================================
# END
# ============================================================


## 元セルindex 81
構文状態：valid


In [ ]:
# ============================================================
# FROZEN COST ACCOUNTING SEMANTICS DIAGNOSTIC
#
# Purpose:
#   Historical 2026 trades 300件から、
#   実際に使用されていた cost / sizing の計算順序を確定する。
#
# NO TRAINING
# NO OPTIMIZATION
# NO STRATEGY CHANGE
# NO ORDER
# ============================================================

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 0. CONSTANTS
# ============================================================

FROZEN_BASE_COST = 4e-05
EXPECTED_TRADES = 300

ABS_TOL = 1e-12


# ============================================================
# 1. HELPERS
# ============================================================

def _find_col(df, candidates):
    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for c in candidates:
        key = str(c).strip().lower()
        if key in lookup:
            return lookup[key]

    return None


def _num(x):
    return pd.to_numeric(
        pd.Series(x).reset_index(drop=True),
        errors="coerce"
    )


def _max_abs(x):
    x = pd.to_numeric(
        pd.Series(x),
        errors="coerce"
    )
    x = x.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(x) == 0:
        return np.nan

    return float(x.abs().max())


# ============================================================
# 2. LOAD HISTORICAL REFERENCE
# ============================================================

print("=" * 110)
print("FROZEN COST ACCOUNTING SEMANTICS DIAGNOSTIC")
print("=" * 110)

src = globals().get(
    "HISTORICAL_REPLAY_LIVE_TRADES"
)

if not isinstance(src, pd.DataFrame):

    COST_ACCOUNTING_DIAGNOSTIC_OK = False
    COST_ACCOUNTING_MODE = None

    print("STOP")
    print(
        "HISTORICAL_REPLAY_LIVE_TRADES が見つかりません。"
    )

else:

    df = src.copy().reset_index(drop=True)

    print("Historical rows:", len(df))

    gross_col = _find_col(
        df,
        ["gross_return"]
    )

    net_col = _find_col(
        df,
        ["net_return"]
    )

    size_col = _find_col(
        df,
        ["position_size"]
    )

    missing = []

    if gross_col is None:
        missing.append("gross_return")

    if net_col is None:
        missing.append("net_return")

    if size_col is None:
        missing.append("position_size")


    if len(df) != EXPECTED_TRADES:

        COST_ACCOUNTING_DIAGNOSTIC_OK = False
        COST_ACCOUNTING_MODE = None

        print()
        print("STOP")
        print(
            f"Expected {EXPECTED_TRADES} trades, "
            f"but found {len(df)}."
        )

    elif missing:

        COST_ACCOUNTING_DIAGNOSTIC_OK = False
        COST_ACCOUNTING_MODE = None

        print()
        print("STOP")
        print(
            "Missing columns:",
            missing
        )

    else:

        gross = _num(
            df[gross_col]
        )

        net = _num(
            df[net_col]
        )

        size = _num(
            df[size_col]
        )


        valid = (
            gross.notna()
            &
            net.notna()
            &
            size.notna()
            &
            (size != 0)
        )


        if int(valid.sum()) != EXPECTED_TRADES:

            COST_ACCOUNTING_DIAGNOSTIC_OK = False
            COST_ACCOUNTING_MODE = None

            print()
            print("STOP")
            print(
                "Valid rows:",
                int(valid.sum())
            )

        else:

            gross = gross[valid].reset_index(drop=True)
            net = net[valid].reset_index(drop=True)
            size = size[valid].reset_index(drop=True)


            # =================================================
            # 3. CANDIDATE ACCOUNTING FORMULAS
            # =================================================

            # A:
            # size * gross - fixed cost
            candidate_A = (
                size * gross
                -
                FROZEN_BASE_COST
            )


            # B:
            # size * (gross - cost)
            candidate_B = (
                size
                *
                (
                    gross
                    -
                    FROZEN_BASE_COST
                )
            )


            # C:
            # gross - cost, then size ignored
            candidate_C = (
                gross
                -
                FROZEN_BASE_COST
            )


            # D:
            # size * gross - cost * abs(size)
            # mathematically same as B for positive size,
            # included for explicit exposure interpretation.
            candidate_D = (
                size * gross
                -
                FROZEN_BASE_COST * size.abs()
            )


            candidates = {
                "FIXED_COST_AFTER_SIZING":
                    candidate_A,

                "SIZE_SCALED_COST":
                    candidate_B,

                "UNSIZED_FIXED_COST":
                    candidate_C,

                "EXPOSURE_SCALED_COST":
                    candidate_D,
            }


            rows = []

            for name, predicted in candidates.items():

                diff = predicted - net

                exact_matches = int(
                    np.isclose(
                        predicted,
                        net,
                        atol=ABS_TOL,
                        rtol=0.0
                    ).sum()
                )

                rows.append({
                    "accounting_mode":
                        name,

                    "valid_rows":
                        int(len(net)),

                    "exact_matches_1e-12":
                        exact_matches,

                    "max_abs_diff":
                        _max_abs(diff),

                    "mean_abs_diff":
                        float(
                            diff.abs().mean()
                        ),
                })


            result = pd.DataFrame(rows)

            result = result.sort_values(
                [
                    "exact_matches_1e-12",
                    "max_abs_diff"
                ],
                ascending=[
                    False,
                    True
                ]
            ).reset_index(drop=True)


            print()
            print("=" * 110)
            print("CANDIDATE COST ACCOUNTING")
            print("=" * 110)

            display(result)


            # =================================================
            # 4. DIRECT IMPLIED COST CHECK
            # =================================================

            # If:
            # net = size * (gross - cost)
            #
            # cost =
            # gross - net / size

            implied_unit_cost = (
                gross
                -
                net / size
            )


            implied_total_cost = (
                size * gross
                -
                net
            )


            implied_expected_total_cost = (
                FROZEN_BASE_COST
                *
                size
            )


            unit_cost_max_diff = _max_abs(
                implied_unit_cost
                -
                FROZEN_BASE_COST
            )


            total_cost_max_diff = _max_abs(
                implied_total_cost
                -
                implied_expected_total_cost
            )


            print()
            print("=" * 110)
            print("IMPLIED COST DIAGNOSTIC")
            print("=" * 110)

            print(
                "Frozen base cost:",
                FROZEN_BASE_COST
            )

            print(
                "Median implied UNIT cost:",
                float(
                    implied_unit_cost.median()
                )
            )

            print(
                "Max UNIT cost difference:",
                unit_cost_max_diff
            )

            print()
            print(
                "Median position size:",
                float(size.median())
            )

            print(
                "Min position size:",
                float(size.min())
            )

            print(
                "Max position size:",
                float(size.max())
            )

            print()
            print(
                "Median total portfolio cost:",
                float(
                    implied_total_cost.median()
                )
            )

            print(
                "Max total-cost reproduction diff:",
                total_cost_max_diff
            )


            # =================================================
            # 5. SELECT ONLY AN EXACTLY REPRODUCED CONTRACT
            # =================================================

            perfect = result[
                result[
                    "exact_matches_1e-12"
                ]
                ==
                EXPECTED_TRADES
            ].copy()


            # B and D are algebraically identical here because
            # position_size is positive.
            #
            # We freeze the semantic interpretation as
            # SIZE_SCALED_COST.

            size_scaled_row = result[
                result["accounting_mode"]
                ==
                "SIZE_SCALED_COST"
            ]


            if len(size_scaled_row) != 1:

                COST_ACCOUNTING_DIAGNOSTIC_OK = False
                COST_ACCOUNTING_MODE = None

                print()
                print("=" * 110)
                print("FINAL DECISION")
                print("=" * 110)

                print(
                    "STOP_COST_ACCOUNTING_NOT_IDENTIFIED"
                )

            else:

                exact_count = int(
                    size_scaled_row[
                        "exact_matches_1e-12"
                    ].iloc[0]
                )

                max_diff = float(
                    size_scaled_row[
                        "max_abs_diff"
                    ].iloc[0]
                )


                COST_ACCOUNTING_DIAGNOSTIC_OK = bool(
                    exact_count == EXPECTED_TRADES
                    and
                    max_diff <= ABS_TOL
                )


                if COST_ACCOUNTING_DIAGNOSTIC_OK:

                    COST_ACCOUNTING_MODE = (
                        "SIZE_SCALED_COST"
                    )

                    FROZEN_COST_ACCOUNTING_MODE = (
                        "POSITION_SIZE_X_GROSS_MINUS_COST"
                    )

                    FROZEN_COST_FORMULA = (
                        "net_return = "
                        "position_size * "
                        "(gross_return - base_cost)"
                    )

                    FROZEN_COST_PER_1X = (
                        FROZEN_BASE_COST
                    )


                    print()
                    print("=" * 110)
                    print("FROZEN COST CONTRACT")
                    print("=" * 110)

                    print(
                        "Accounting mode:",
                        COST_ACCOUNTING_MODE
                    )

                    print(
                        "Formula:"
                    )

                    print(
                        "net_return = "
                        "position_size * "
                        "(gross_return - 4e-05)"
                    )

                    print()
                    print(
                        "Base cost @ 1.0x:",
                        FROZEN_BASE_COST
                    )

                    print(
                        "Exact matches:",
                        f"{exact_count}/{EXPECTED_TRADES}"
                    )

                    print(
                        "Max difference:",
                        max_diff
                    )


                    print()
                    print("=" * 110)
                    print("FINAL DECISION")
                    print("=" * 110)

                    print(
                        "COST ACCOUNTING SEMANTICS PASSED:",
                        True
                    )

                    print(
                        "FINAL DECISION:",
                        "PASS_SIZE_SCALED_COST_CONTRACT_FROZEN"
                    )

                    print()
                    print(
                        "Champion/model/features were NOT modified."
                    )

                    print(
                        "Execution accounting semantics only "
                        "has been reconstructed."
                    )

                    print()
                    print(
                        "NEXT STEP:"
                    )

                    print(
                        "Rerun FINAL 2026 PAPER EXECUTION REPLAY "
                        "using:"
                    )

                    print(
                        "net_return = "
                        "position_size * "
                        "(gross_return - FROZEN_BASE_COST)"
                    )

                else:

                    COST_ACCOUNTING_MODE = None

                    print()
                    print("=" * 110)
                    print("FINAL DECISION")
                    print("=" * 110)

                    print(
                        "COST ACCOUNTING SEMANTICS PASSED:",
                        False
                    )

                    print(
                        "FINAL DECISION:",
                        "STOP_COST_FORMULA_NOT_REPRODUCED"
                    )

                    print()
                    print(
                        "Do NOT change the strategy."
                    )

                    print(
                        "Inspect historical net-return semantics."
                    )


# ============================================================
# 6. SAVE REPORT VARIABLES
# ============================================================

globals()[
    "COST_ACCOUNTING_DIAGNOSTIC_OK"
] = globals().get(
    "COST_ACCOUNTING_DIAGNOSTIC_OK",
    False
)

globals()[
    "COST_ACCOUNTING_MODE"
] = globals().get(
    "COST_ACCOUNTING_MODE",
    None
)


print()
print("=" * 110)
print("CELL COMPLETE")
print("=" * 110)

print(
    "Passed:",
    globals().get(
        "COST_ACCOUNTING_DIAGNOSTIC_OK"
    )
)

print(
    "Mode:",
    globals().get(
        "COST_ACCOUNTING_MODE"
    )
)


## 元セルindex 82
構文状態：original_syntax_error: expected 'except' or 'finally' block at line 2239


In [ ]:
# ============================================================
# FINAL 2026 PAPER EXECUTION REPLAY
# Frozen Champion Execution Contract
#
# PURPOSE
# ------------------------------------------------------------
# 2026 historical 300 trades を Paper Execution Engine として
# 時系列順に再実行し、Frozen Backtest と完全一致するか確認する。
#
# NO TRAINING
# NO OPTIMIZATION
# NO FEATURE CHANGE
# NO MODEL CHANGE
# NO REAL ORDER
#
# Frozen contract:
#   signal -> entry : +15 min
#   entry -> exit   : +30 min
#   entry price     : OPEN_AT_TIME
#   exit price      : PREVIOUS_BAR_CLOSE
#   gross return    : SIGNED_SIMPLE_RETURN
#   net return      : position_size * (gross_return - 4e-05)
#   overlap         : PROHIBITED
#   event priority  : EXIT before ENTRY at same timestamp
# ============================================================

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 0. FROZEN CONSTANTS
# ============================================================

EXPECTED_TRADES = 300

FROZEN_BASE_COST = 4e-05

SIGNAL_TO_ENTRY_MIN = 15
ENTRY_TO_EXIT_MIN = 30

STARTING_EQUITY = 1_000_000.0

FLOAT_TOL = 1e-12
PRICE_RETURN_TOL = 1e-10


# ============================================================
# 1. HELPERS
# ============================================================

def _line(title):
    print()
    print("=" * 110)
    print(title)
    print("=" * 110)


def _find_col(df, candidates):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        key = str(candidate).strip().lower()

        if key in lookup:
            return lookup[key]

    return None


def _to_utc(values):

    return pd.to_datetime(
        values,
        utc=True,
        errors="coerce"
    )


def _numeric(values):

    return pd.to_numeric(
        pd.Series(values).reset_index(drop=True),
        errors="coerce"
    )


def _max_abs(values):

    s = pd.to_numeric(
        pd.Series(values),
        errors="coerce"
    )

    s = s.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(s) == 0:
        return np.nan

    return float(
        np.abs(s).max()
    )


def _profit_factor(returns):

    r = np.asarray(
        returns,
        dtype=float
    )

    wins = r[r > 0].sum()

    losses = -r[r < 0].sum()

    if losses <= 0:
        return np.inf

    return float(
        wins / losses
    )


def _performance(returns):

    r = np.asarray(
        returns,
        dtype=float
    )

    if len(r) == 0:

        return {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "profit_factor": np.nan,
            "growth": np.nan,
            "max_dd": np.nan,
            "return_to_dd": np.nan,
        }

    equity = np.cumprod(
        1.0 + r
    )

    peak = np.maximum.accumulate(
        equity
    )

    dd = equity / peak - 1.0

    growth = float(
        equity[-1] - 1.0
    )

    max_dd = float(
        np.min(dd)
    )

    return_to_dd = (
        growth / abs(max_dd)
        if max_dd < 0
        else np.inf
    )

    return {
        "trades":
            int(len(r)),

        "win_rate":
            float(np.mean(r > 0)),

        "avg_return":
            float(np.mean(r)),

        "profit_factor":
            _profit_factor(r),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            float(return_to_dd),
    }


def _normalize_canonical(df):

    if not isinstance(df, pd.DataFrame):
        return None

    lower = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    if "open" not in lower or "close" not in lower:
        return None

    # ------------------------------------------
    # Timestamp source
    # ------------------------------------------

    if isinstance(
        df.index,
        pd.DatetimeIndex
    ):

        idx = pd.to_datetime(
            df.index,
            utc=True,
            errors="coerce"
        )

    else:

        time_col = None

        for candidate in [
            "timestamp",
            "datetime",
            "time",
            "date"
        ]:

            if candidate in lower:
                time_col = lower[candidate]
                break

        if time_col is None:
            return None

        idx = pd.to_datetime(
            df[time_col],
            utc=True,
            errors="coerce"
        )


    out = pd.DataFrame(
        {
            "open":
                pd.to_numeric(
                    df[lower["open"]].values,
                    errors="coerce"
                ),

            "close":
                pd.to_numeric(
                    df[lower["close"]].values,
                    errors="coerce"
                ),
        },
        index=idx
    )


    out = out[
        ~out.index.isna()
    ].copy()

    out = out[
        out["open"].notna()
        &
        out["close"].notna()
    ].copy()

    out = out.sort_index()


    # Canonical history must be unique.
    if out.index.has_duplicates:
        return None

    return out


# ============================================================
# 2. DEFAULT OUTPUT STATE
# ============================================================

FINAL_2026_PAPER_REPLAY_PASSED = False
FINAL_2026_PAPER_REPLAY_DECISION = "NOT_RUN"

FINAL_2026_PAPER_REPLAY = pd.DataFrame()
FINAL_2026_PAPER_REPLAY_CHECKS = pd.DataFrame()
FINAL_2026_PAPER_REPLAY_ERRORS = pd.DataFrame()

SELECTED_CANONICAL_SOURCE = None


# ============================================================
# 3. MAIN
# ============================================================

try:

    _line(
        "FINAL 2026 PAPER EXECUTION REPLAY"
    )

    print(
        "Frozen cost:",
        FROZEN_BASE_COST
    )

    print(
        "Entry:",
        "OPEN_AT_TIME"
    )

    print(
        "Exit:",
        "PREVIOUS_BAR_CLOSE"
    )

    print(
        "Holding:",
        ENTRY_TO_EXIT_MIN,
        "minutes"
    )

    print(
        "Event priority:",
        "EXIT -> ENTRY"
    )


    # ========================================================
    # 4. VERIFY COST CONTRACT
    # ========================================================

    _line(
        "FROZEN COST CONTRACT"
    )

    prior_cost_ok = bool(
        globals().get(
            "COST_ACCOUNTING_DIAGNOSTIC_OK",
            False
        )
    )

    prior_cost_mode = globals().get(
        "COST_ACCOUNTING_MODE",
        None
    )

    print(
        "Prior diagnostic passed:",
        prior_cost_ok
    )

    print(
        "Prior mode:",
        prior_cost_mode
    )

    print(
        "Formula:"
    )

    print(
        "net_return = "
        "position_size * "
        "(gross_return - 4e-05)"
    )


    # ========================================================
    # 5. LOAD HISTORICAL REFERENCE
    # ========================================================

    _line(
        "HISTORICAL REFERENCE"
    )

    reference_obj = globals().get(
        "HISTORICAL_REPLAY_LIVE_TRADES"
    )

    if not isinstance(
        reference_obj,
        pd.DataFrame
    ):

        FINAL_2026_PAPER_REPLAY_DECISION = (
            "STOP_HISTORICAL_REFERENCE_NOT_FOUND"
        )

        print(
            "STOP:"
        )

        print(
            "HISTORICAL_REPLAY_LIVE_TRADES "
            "was not found."
        )

    else:

        ref = (
            reference_obj
            .copy()
            .reset_index(drop=True)
        )

        print(
            "Rows:",
            len(ref)
        )


        # ====================================================
        # 6. IDENTIFY REQUIRED COLUMNS
        # ====================================================

        signal_col = _find_col(
            ref,
            [
                "signal_time",
                "_diag_signal_time"
            ]
        )

        entry_col = _find_col(
            ref,
            ["entry_time"]
        )

        exit_col = _find_col(
            ref,
            ["exit_time"]
        )

        side_col = _find_col(
            ref,
            ["side"]
        )

        size_col = _find_col(
            ref,
            ["position_size"]
        )

        gross_col = _find_col(
            ref,
            ["gross_return"]
        )

        net_col = _find_col(
            ref,
            ["net_return"]
        )


        required = {
            "signal_time":
                signal_col,

            "entry_time":
                entry_col,

            "exit_time":
                exit_col,

            "side":
                side_col,

            "position_size":
                size_col,

            "gross_return":
                gross_col,

            "net_return":
                net_col,
        }


        missing = [
            k
            for k, v in required.items()
            if v is None
        ]


        if missing:

            FINAL_2026_PAPER_REPLAY_DECISION = (
                "STOP_MISSING_REFERENCE_COLUMNS"
            )

            print(
                "STOP:"
            )

            print(
                "Missing columns:",
                missing
            )

        elif len(ref) != EXPECTED_TRADES:

            FINAL_2026_PAPER_REPLAY_DECISION = (
                "STOP_REFERENCE_TRADE_COUNT"
            )

            print(
                "STOP:"
            )

            print(
                "Expected:",
                EXPECTED_TRADES
            )

            print(
                "Found:",
                len(ref)
            )

        else:

            # =================================================
            # 7. NORMALIZE REFERENCE
            # =================================================

            reference = pd.DataFrame()

            reference[
                "trade_id"
            ] = np.arange(
                len(ref),
                dtype=int
            )

            reference[
                "signal_time"
            ] = _to_utc(
                ref[signal_col]
            )

            reference[
                "entry_time"
            ] = _to_utc(
                ref[entry_col]
            )

            reference[
                "exit_time"
            ] = _to_utc(
                ref[exit_col]
            )

            reference[
                "side"
            ] = (
                ref[side_col]
                .astype(str)
                .str.upper()
                .str.strip()
                .reset_index(drop=True)
            )

            reference[
                "position_size"
            ] = _numeric(
                ref[size_col]
            )

            reference[
                "expected_gross_return"
            ] = _numeric(
                ref[gross_col]
            )

            reference[
                "expected_net_return"
            ] = _numeric(
                ref[net_col]
            )


            valid_reference = bool(
                reference[
                    [
                        "signal_time",
                        "entry_time",
                        "exit_time",
                        "position_size",
                        "expected_gross_return",
                        "expected_net_return",
                    ]
                ]
                .notna()
                .all()
                .all()
            )

            valid_sides = bool(
                reference["side"]
                .isin(
                    ["BUY", "SELL"]
                )
                .all()
            )


            # =================================================
            # 8. TIMING CONTRACT
            # =================================================

            signal_to_entry = (
                reference["entry_time"]
                -
                reference["signal_time"]
            )

            entry_to_exit = (
                reference["exit_time"]
                -
                reference["entry_time"]
            )


            timing_signal_entry_ok = bool(
                (
                    signal_to_entry
                    ==
                    pd.Timedelta(
                        minutes=SIGNAL_TO_ENTRY_MIN
                    )
                ).all()
            )

            timing_entry_exit_ok = bool(
                (
                    entry_to_exit
                    ==
                    pd.Timedelta(
                        minutes=ENTRY_TO_EXIT_MIN
                    )
                ).all()
            )


            duplicate_signals = int(
                reference[
                    "signal_time"
                ].duplicated().sum()
            )


            sorted_ref = reference.sort_values(
                "entry_time"
            ).reset_index(drop=True)


            previous_exit = (
                sorted_ref[
                    "exit_time"
                ]
                .shift(1)
            )


            historical_overlap = int(
                (
                    sorted_ref[
                        "entry_time"
                    ]
                    <
                    previous_exit
                )
                .fillna(False)
                .sum()
            )


            print()
            print(
                "Valid reference:",
                valid_reference
            )

            print(
                "Valid sides:",
                valid_sides
            )

            print(
                "Signal -> Entry = 15m:",
                timing_signal_entry_ok
            )

            print(
                "Entry -> Exit = 30m:",
                timing_entry_exit_ok
            )

            print(
                "Duplicate signals:",
                duplicate_signals
            )

            print(
                "Historical overlap:",
                historical_overlap
            )


            # =================================================
            # 9. FIND CANONICAL HISTORY
            # =================================================

            _line(
                "CANONICAL HISTORY SELECTION"
            )


            preferred_names = [
                "PRODUCTION_CANONICAL_HISTORY",
                "bars_rebuilt_clean",
                "bars_clean_candidate",
                "bars",
            ]


            candidates = []


            # Named candidates first
            for name in preferred_names:

                obj = globals().get(
                    name
                )

                normalized = (
                    _normalize_canonical(obj)
                )

                if normalized is not None:

                    candidates.append(
                        (
                            name,
                            normalized
                        )
                    )


            # Search other notebook DataFrames only if needed
            seen_names = {
                name
                for name, _ in candidates
            }


            for name, obj in list(
                globals().items()
            ):

                if name in seen_names:
                    continue

                if not isinstance(
                    obj,
                    pd.DataFrame
                ):
                    continue

                if len(obj) < 100_000:
                    continue

                normalized = (
                    _normalize_canonical(obj)
                )

                if normalized is None:
                    continue

                candidates.append(
                    (
                        name,
                        normalized
                    )
                )


            candidate_reports = []


            for name, can in candidates:

                entry_times = pd.DatetimeIndex(
                    reference[
                        "entry_time"
                    ]
                )

                exit_price_times = pd.DatetimeIndex(
                    reference[
                        "exit_time"
                    ]
                    -
                    pd.Timedelta(
                        minutes=15
                    )
                )


                entry_prices = (
                    can["open"]
                    .reindex(
                        entry_times
                    )
                    .to_numpy(
                        dtype=float
                    )
                )

                exit_prices = (
                    can["close"]
                    .reindex(
                        exit_price_times
                    )
                    .to_numpy(
                        dtype=float
                    )
                )


                available = (
                    np.isfinite(
                        entry_prices
                    )
                    &
                    np.isfinite(
                        exit_prices
                    )
                )


                raw_move = (
                    exit_prices
                    /
                    entry_prices
                    -
                    1.0
                )


                sides = (
                    reference[
                        "side"
                    ]
                    .to_numpy()
                )


                replay_gross = np.where(
                    sides == "BUY",
                    raw_move,
                    -raw_move
                )


                expected_gross = (
                    reference[
                        "expected_gross_return"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )


                diff = np.abs(
                    replay_gross
                    -
                    expected_gross
                )


                exact = (
                    available
                    &
                    np.isfinite(diff)
                    &
                    (
                        diff
                        <= PRICE_RETURN_TOL
                    )
                )


                candidate_reports.append(
                    {
                        "source":
                            name,

                        "rows":
                            len(can),

                        "available_prices":
                            int(
                                available.sum()
                            ),

                        "gross_matches":
                            int(
                                exact.sum()
                            ),

                        "max_gross_diff":
                            (
                                float(
                                    np.nanmax(
                                        diff[
                                            available
                                        ]
                                    )
                                )
                                if available.any()
                                else np.nan
                            ),
                    }
                )


            candidate_report_df = pd.DataFrame(
                candidate_reports
            )


            if len(
                candidate_report_df
            ) > 0:

                candidate_report_df = (
                    candidate_report_df
                    .sort_values(
                        [
                            "gross_matches",
                            "available_prices"
                        ],
                        ascending=[
                            False,
                            False
                        ]
                    )
                    .reset_index(
                        drop=True
                    )
                )

                display(
                    candidate_report_df.head(
                        10
                    )
                )


            exact_sources = []

            for name, can in candidates:

                row = candidate_report_df[
                    candidate_report_df[
                        "source"
                    ]
                    ==
                    name
                ]

                if len(row) != 1:
                    continue

                if (
                    int(
                        row[
                            "available_prices"
                        ].iloc[0]
                    )
                    ==
                    EXPECTED_TRADES
                    and
                    int(
                        row[
                            "gross_matches"
                        ].iloc[0]
                    )
                    ==
                    EXPECTED_TRADES
                ):

                    exact_sources.append(
                        (
                            name,
                            can
                        )
                    )


            if len(
                exact_sources
            ) == 0:

                FINAL_2026_PAPER_REPLAY_DECISION = (
                    "STOP_CANONICAL_HISTORY_NOT_REPRODUCED"
                )

                print()
                print(
                    "STOP:"
                )

                print(
                    "No canonical history source "
                    "reproduced all 300 historical returns."
                )

            else:

                # Preferred named source wins when exact.
                selected_name = None
                canonical = None

                for preferred in preferred_names:

                    for name, can in exact_sources:

                        if name == preferred:

                            selected_name = (
                                name
                            )

                            canonical = (
                                can
                            )

                            break

                    if canonical is not None:
                        break


                if canonical is None:

                    selected_name, canonical = (
                        exact_sources[0]
                    )


                SELECTED_CANONICAL_SOURCE = (
                    selected_name
                )


                print()
                print(
                    "Selected:",
                    SELECTED_CANONICAL_SOURCE
                )

                print(
                    "Rows:",
                    len(canonical)
                )

                print(
                    "Period:",
                    canonical.index.min(),
                    "->",
                    canonical.index.max()
                )


                # =================================================
                # 10. BUILD EXECUTION EVENTS
                # =================================================

                _line(
                    "BUILD PAPER EXECUTION EVENTS"
                )


                events = []

                for row in reference.itertuples(
                    index=False
                ):

                    events.append(
                        {
                            "event_time":
                                row.entry_time,

                            "priority":
                                1,

                            "event":
                                "ENTRY",

                            "trade_id":
                                int(
                                    row.trade_id
                                ),
                        }
                    )

                    events.append(
                        {
                            "event_time":
                                row.exit_time,

                            "priority":
                                0,

                            "event":
                                "EXIT",

                            "trade_id":
                                int(
                                    row.trade_id
                                ),
                        }
                    )


                events = pd.DataFrame(
                    events
                )

                events = (
                    events
                    .sort_values(
                        [
                            "event_time",
                            "priority",
                            "trade_id"
                        ]
                    )
                    .reset_index(
                        drop=True
                    )
                )


                print(
                    "Events:",
                    len(events)
                )

                print(
                    "ENTRY events:",
                    int(
                        (
                            events["event"]
                            ==
                            "ENTRY"
                        ).sum()
                    )
                )

                print(
                    "EXIT events:",
                    int(
                        (
                            events["event"]
                            ==
                            "EXIT"
                        ).sum()
                    )
                )

                print(
                    "Same timestamp rule:",
                    "EXIT FIRST"
                )


                # =================================================
                # 11. PAPER EXECUTION ENGINE
                # =================================================

                _line(
                    "RUNNING PAPER EXECUTION ENGINE"
                )


                reference_by_id = (
                    reference
                    .set_index(
                        "trade_id"
                    )
                )


                open_position = None

                equity = float(
                    STARTING_EQUITY
                )

                execution_records = []
                engine_errors = []


                for ev in events.itertuples(
                    index=False
                ):

                    trade_id = int(
                        ev.trade_id
                    )

                    event_time = (
                        pd.Timestamp(
                            ev.event_time
                        )
                    )

                    ref_row = (
                        reference_by_id
                        .loc[
                            trade_id
                        ]
                    )


                    # =========================================
                    # EXIT
                    # =========================================

                    if ev.event == "EXIT":

                        if open_position is None:

                            engine_errors.append(
                                {
                                    "event_time":
                                        event_time,

                                    "error":
                                        "EXIT_WITHOUT_POSITION",

                                    "trade_id":
                                        trade_id,
                                }
                            )

                            continue


                        if (
                            int(
                                open_position[
                                    "trade_id"
                                ]
                            )
                            !=
                            trade_id
                        ):

                            engine_errors.append(
                                {
                                    "event_time":
                                        event_time,

                                    "error":
                                        "WRONG_POSITION_AT_EXIT",

                                    "trade_id":
                                        trade_id,

                                    "open_trade_id":
                                        int(
                                            open_position[
                                                "trade_id"
                                            ]
                                        ),
                                }
                            )

                            continue


                        exit_price_time = (
                            event_time
                            -
                            pd.Timedelta(
                                minutes=15
                            )
                        )


                        if (
                            exit_price_time
                            not in
                            canonical.index
                        ):

                            engine_errors.append(
                                {
                                    "event_time":
                                        event_time,

                                    "error":
                                        "MISSING_EXIT_PRICE",

                                    "trade_id":
                                        trade_id,
                                }
                            )

                            continue


                        exit_price = float(
                            canonical.at[
                                exit_price_time,
                                "close"
                            ]
                        )


                        entry_price = float(
                            open_position[
                                "entry_price"
                            ]
                        )


                        raw_return = (
                            exit_price
                            /
                            entry_price
                            -
                            1.0
                        )


                        if (
                            open_position[
                                "side"
                            ]
                            ==
                            "BUY"
                        ):

                            gross_return = (
                                raw_return
                            )

                        else:

                            gross_return = (
                                -raw_return
                            )


                        position_size = float(
                            open_position[
                                "position_size"
                            ]
                        )


                        # -------------------------------------
                        # FROZEN COST ACCOUNTING
                        # -------------------------------------

                        net_return = (
                            position_size
                            *
                            (
                                gross_return
                                -
                                FROZEN_BASE_COST
                            )
                        )


                        equity_before = (
                            equity
                        )

                        equity = (
                            equity
                            *
                            (
                                1.0
                                +
                                net_return
                            )
                        )


                        execution_records.append(
                            {
                                "trade_id":
                                    trade_id,

                                "signal_time":
                                    open_position[
                                        "signal_time"
                                    ],

                                "entry_time":
                                    open_position[
                                        "entry_time"
                                    ],

                                "exit_time":
                                    event_time,

                                "side":
                                    open_position[
                                        "side"
                                    ],

                                "position_size":
                                    position_size,

                                "entry_price":
                                    entry_price,

                                "exit_price":
                                    exit_price,

                                "gross_return":
                                    gross_return,

                                "net_return":
                                    net_return,

                                "equity_before":
                                    equity_before,

                                "equity_after":
                                    equity,

                                "expected_gross_return":
                                    float(
                                        ref_row[
                                            "expected_gross_return"
                                        ]
                                    ),

                                "expected_net_return":
                                    float(
                                        ref_row[
                                            "expected_net_return"
                                        ]
                                    ),
                            }
                        )


                        open_position = None


                    # =========================================
                    # ENTRY
                    # =========================================

                    elif ev.event == "ENTRY":

                        if open_position is not None:

                            engine_errors.append(
                                {
                                    "event_time":
                                        event_time,

                                    "error":
                                        "OVERLAP_ENTRY",

                                    "trade_id":
                                        trade_id,

                                    "open_trade_id":
                                        int(
                                            open_position[
                                                "trade_id"
                                            ]
                                        ),
                                }
                            )

                            continue


                        if (
                            event_time
                            not in
                            canonical.index
                        ):

                            engine_errors.append(
                                {
                                    "event_time":
                                        event_time,

                                    "error":
                                        "MISSING_ENTRY_PRICE",

                                    "trade_id":
                                        trade_id,
                                }
                            )

                            continue


                        entry_price = float(
                            canonical.at[
                                event_time,
                                "open"
                            ]
                        )


                        open_position = {
                            "trade_id":
                                trade_id,

                            "signal_time":
                                ref_row[
                                    "signal_time"
                                ],

                            "entry_time":
                                event_time,

                            "side":
                                ref_row[
                                    "side"
                                ],

                            "position_size":
                                float(
                                    ref_row[
                                        "position_size"
                                    ]
                                ),

                            "entry_price":
                                entry_price,
                        }


                # =================================================
                # 12. RESULTS
                # =================================================

                replay = pd.DataFrame(
                    execution_records
                )

                errors = pd.DataFrame(
                    engine_errors
                )


                FINAL_2026_PAPER_REPLAY = (
                    replay.copy()
                )

                FINAL_2026_PAPER_REPLAY_ERRORS = (
                    errors.copy()
                )


                print(
                    "Expected trades:",
                    EXPECTED_TRADES
                )

                print(
                    "Executed trades:",
                    len(replay)
                )

                print(
                    "Engine errors:",
                    len(errors)
                )

                print(
                    "Open position after replay:",
                    open_position is not None
                )

                print(
                    "Ending equity:",
                    f"{equity:,.6f}"
                )


                # =================================================
                # 13. TRADE-BY-TRADE PARITY
                # =================================================

                _line(
                    "TRADE-BY-TRADE PARITY"
                )


                if len(replay) > 0:

                    replay[
                        "gross_diff"
                    ] = (
                        replay[
                            "gross_return"
                        ]
                        -
                        replay[
                            "expected_gross_return"
                        ]
                    )

                    replay[
                        "net_diff"
                    ] = (
                        replay[
                            "net_return"
                        ]
                        -
                        replay[
                            "expected_net_return"
                        ]
                    )


                    max_gross_diff = (
                        _max_abs(
                            replay[
                                "gross_diff"
                            ]
                        )
                    )

                    max_net_diff = (
                        _max_abs(
                            replay[
                                "net_diff"
                            ]
                        )
                    )


                    gross_match = bool(
                        max_gross_diff
                        <=
                        PRICE_RETURN_TOL
                    )

                    net_match = bool(
                        max_net_diff
                        <=
                        FLOAT_TOL
                    )

                else:

                    max_gross_diff = (
                        np.nan
                    )

                    max_net_diff = (
                        np.nan
                    )

                    gross_match = False
                    net_match = False


                print(
                    "Max gross-return diff:",
                    max_gross_diff
                )

                print(
                    "Gross return parity:",
                    gross_match
                )

                print(
                    "Max net-return diff:",
                    max_net_diff
                )

                print(
                    "Net return parity:",
                    net_match
                )


                # =================================================
                # 14. COST CONTRACT RECHECK
                # =================================================

                if len(replay) > 0:

                    cost_formula_check = (
                        replay[
                            "position_size"
                        ]
                        *
                        (
                            replay[
                                "gross_return"
                            ]
                            -
                            FROZEN_BASE_COST
                        )
                    )


                    cost_formula_diff = (
                        cost_formula_check
                        -
                        replay[
                            "net_return"
                        ]
                    )


                    cost_contract_match = bool(
                        _max_abs(
                            cost_formula_diff
                        )
                        <=
                        FLOAT_TOL
                    )

                else:

                    cost_contract_match = (
                        False
                    )


                print(
                    "Cost accounting parity:",
                    cost_contract_match
                )


                # =================================================
                # 15. PERFORMANCE PARITY
                # =================================================

                _line(
                    "PERFORMANCE PARITY"
                )


                expected_perf = (
                    _performance(
                        reference[
                            "expected_net_return"
                        ]
                    )
                )


                replay_perf = (
                    _performance(
                        replay[
                            "net_return"
                        ]
                        if len(replay)
                        else []
                    )
                )


                performance_rows = []

                for metric in [
                    "trades",
                    "win_rate",
                    "avg_return",
                    "profit_factor",
                    "growth",
                    "max_dd",
                    "return_to_dd",
                ]:

                    expected_value = (
                        expected_perf[
                            metric
                        ]
                    )

                    replay_value = (
                        replay_perf[
                            metric
                        ]
                    )


                    if (
                        isinstance(
                            expected_value,
                            (int, np.integer)
                        )
                        and
                        isinstance(
                            replay_value,
                            (int, np.integer)
                        )
                    ):

                        diff = (
                            replay_value
                            -
                            expected_value
                        )

                    else:

                        try:

                            diff = float(
                                replay_value
                                -
                                expected_value
                            )

                        except Exception:

                            diff = np.nan


                    performance_rows.append(
                        {
                            "metric":
                                metric,

                            "expected":
                                expected_value,

                            "replay":
                                replay_value,

                            "diff":
                                diff,
                        }
                    )


                performance_df = pd.DataFrame(
                    performance_rows
                )

                display(
                    performance_df
                )


                performance_numeric = [
                    "win_rate",
                    "avg_return",
                    "profit_factor",
                    "growth",
                    "max_dd",
                    "return_to_dd",
                ]


                performance_ok = True


                for metric in performance_numeric:

                    a = expected_perf[
                        metric
                    ]

                    b = replay_perf[
                        metric
                    ]


                    if (
                        np.isinf(a)
                        and
                        np.isinf(b)
                    ):
                        continue


                    if (
                        not np.isfinite(a)
                        or
                        not np.isfinite(b)
                    ):

                        performance_ok = (
                            False
                        )

                        break


                    if abs(
                        float(a)
                        -
                        float(b)
                    ) > 1e-10:

                        performance_ok = (
                            False
                        )

                        break


                performance_ok = bool(
                    performance_ok
                    and
                    replay_perf[
                        "trades"
                    ]
                    ==
                    EXPECTED_TRADES
                )


                # =================================================
                # 16. FINAL CHECKS
                # =================================================

                _line(
                    "FINAL CHECKS"
                )


                checks = {
                    "prior_cost_contract":
                        prior_cost_ok,

                    "reference_rows_300":
                        len(reference)
                        ==
                        EXPECTED_TRADES,

                    "reference_valid":
                        valid_reference,

                    "side_values_valid":
                        valid_sides,

                    "signal_to_entry_15m":
                        timing_signal_entry_ok,

                    "entry_to_exit_30m":
                        timing_entry_exit_ok,

                    "duplicate_signals_zero":
                        duplicate_signals
                        ==
                        0,

                    "historical_overlap_zero":
                        historical_overlap
                        ==
                        0,

                    "canonical_source_found":
                        canonical
                        is not None,

                    "executed_trades_300":
                        len(replay)
                        ==
                        EXPECTED_TRADES,

                    "engine_errors_zero":
                        len(errors)
                        ==
                        0,

                    "final_position_flat":
                        open_position
                        is None,

                    "gross_return_parity":
                        gross_match,

                    "cost_accounting_parity":
                        cost_contract_match,

                    "net_return_parity":
                        net_match,

                    "performance_parity":
                        performance_ok,
                }


                check_df = pd.DataFrame(
                    {
                        "check":
                            list(
                                checks.keys()
                            ),

                        "passed":
                            list(
                                checks.values()
                            ),
                    }
                )


                FINAL_2026_PAPER_REPLAY_CHECKS = (
                    check_df.copy()
                )


                display(
                    check_df
                )


                final_pass = bool(
                    all(
                        checks.values()
                    )
                )


                FINAL_2026_PAPER_REPLAY_PASSED = (
                    final_pass
                )


                # =================================================
                # 17. SAVE NOTEBOOK OUTPUTS
                # =================================================

                FINAL_2026_PAPER_REPLAY = (
                    replay.copy()
                )


                FINAL_2026_PAPER_REPLAY_PERFORMANCE = (
                    performance_df.copy()
                )


                FINAL_2026_PAPER_REPLAY_EXPECTED_PERF = (
                    expected_perf
                )


                FINAL_2026_PAPER_REPLAY_ACTUAL_PERF = (
                    replay_perf
                )


                # =================================================
                # 18. FINAL DECISION
                # =================================================

                _line(
                    "FINAL 2026 PAPER EXECUTION REPLAY REPORT"
                )


                print(
                    "Champion:",
                    "BASE_PLUS_REGIME"
                )

                print(
                    "Expected trades:",
                    EXPECTED_TRADES
                )

                print(
                    "Executed trades:",
                    len(replay)
                )

                print(
                    "Missing trades:",
                    (
                        EXPECTED_TRADES
                        -
                        len(replay)
                    )
                )

                print(
                    "Execution errors:",
                    len(errors)
                )

                print(
                    "Maximum gross diff:",
                    max_gross_diff
                )

                print(
                    "Maximum net diff:",
                    max_net_diff
                )

                print(
                    "Starting equity:",
                    f"{STARTING_EQUITY:,.2f}"
                )

                print(
                    "Ending equity:",
                    f"{equity:,.2f}"
                )


                print()
                print(
                    "FINAL 2026 PAPER EXECUTION "
                    "REPLAY PASSED:",
                    final_pass
                )


                if final_pass:

                    FINAL_2026_PAPER_REPLAY_DECISION = (
                        "PASS_READY_FOR_FORWARD_PAPER_TRADING"
                    )

                    print()
                    print(
                        "FINAL DECISION:",
                        FINAL_2026_PAPER_REPLAY_DECISION
                    )

                    print()
                    print(
                        "The Frozen Champion was NOT modified."
                    )

                    print(
                        "Historical execution semantics "
                        "were reproduced trade-by-trade."
                    )

                    print(
                        "300 expected trades -> "
                        "300 executed trades."
                    )

                    print(
                        "No duplicate, overlap, "
                        "or accounting mismatch."
                    )

                    print()
                    print(
                        "NEXT STEP:"
                    )

                    print(
                        "BUILD FORWARD PAPER TRADING RUNNER."
                    )

                else:

                    FINAL_2026_PAPER_REPLAY_DECISION = (
                        "STOP_REPLAY_PARITY_FAILED"
                    )

                    print()
                    print(
                        "FINAL DECISION:",
                        FINAL_2026_PAPER_REPLAY_DECISION
                    )

                    print()
                    print(
                        "STOP."
                    )

                    print(
                        "Do not start Forward Paper Trading."
                    )

                    print(
                        "Inspect only the failed checks above."
                    )


                if len(errors) > 0:

                    print()
                    print(
                        "First execution errors:"
                    )

                    display(
                        errors.head(
                            20
                        )
                    )


# ============================================================
# 19. GLOBAL SAVE
# ============================================================

globals()[
    "FINAL_2026_PAPER_REPLAY_PASSED"
] = FINAL_2026_PAPER_REPLAY_PASSED

globals()[
    "FINAL_2026_PAPER_REPLAY_DECISION"
] = FINAL_2026_PAPER_REPLAY_DECISION

globals()[
    "FINAL_2026_PAPER_REPLAY"
] = FINAL_2026_PAPER_REPLAY

globals()[
    "FINAL_2026_PAPER_REPLAY_CHECKS"
] = FINAL_2026_PAPER_REPLAY_CHECKS

globals()[
    "FINAL_2026_PAPER_REPLAY_ERRORS"
] = FINAL_2026_PAPER_REPLAY_ERRORS

globals()[
    "SELECTED_CANONICAL_SOURCE"
] = SELECTED_CANONICAL_SOURCE


except Exception as e:

    FINAL_2026_PAPER_REPLAY_PASSED = False

    FINAL_2026_PAPER_REPLAY_DECISION = (
        "STOP_UNEXPECTED_REPLAY_EXCEPTION"
    )

    globals()[
        "FINAL_2026_PAPER_REPLAY_PASSED"
    ] = False

    globals()[
        "FINAL_2026_PAPER_REPLAY_DECISION"
    ] = FINAL_2026_PAPER_REPLAY_DECISION

    print()
    print("=" * 110)
    print("CONTROLLED STOP")
    print("=" * 110)

    print(
        "Type:",
        type(e).__name__
    )

    print(
        "Message:",
        str(e)
    )

    print()
    print(
        "No real order was sent."
    )

    print(
        "Champion/model/features were not modified."
    )


# ============================================================
# 20. CELL COMPLETE
# ============================================================

_line(
    "CELL COMPLETE"
)

print(
    "Passed:",
    FINAL_2026_PAPER_REPLAY_PASSED
)

print(
    "Decision:",
    FINAL_2026_PAPER_REPLAY_DECISION
)

print(
    "Canonical source:",
    SELECTED_CANONICAL_SOURCE
)


## 元セルindex 83
構文状態：valid


In [ ]:
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# FINAL 2026 PAPER EXECUTION REPLAY
# ============================================================

EXPECTED_TRADES = 300
BASE_COST = 4e-05
STARTING_EQUITY = 1_000_000.0

RET_TOL = 1e-10
NET_TOL = 1e-12


# ============================================================
# HELPERS
# ============================================================

def banner(text):
    print()
    print("=" * 100)
    print(text)
    print("=" * 100)


def find_col(df, names):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for name in names:

        key = name.lower()

        if key in lookup:
            return lookup[key]

    return None


def normalize_canonical(df):

    if not isinstance(df, pd.DataFrame):
        return None

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    if "open" not in lookup or "close" not in lookup:
        return None

    # ----------------------------------------
    # timestamp
    # ----------------------------------------

    if isinstance(df.index, pd.DatetimeIndex):

        idx = pd.to_datetime(
            df.index,
            utc=True,
            errors="coerce"
        )

    else:

        time_col = find_col(
            df,
            [
                "timestamp",
                "datetime",
                "time",
                "date"
            ]
        )

        if time_col is None:
            return None

        idx = pd.to_datetime(
            df[time_col],
            utc=True,
            errors="coerce"
        )

    # ----------------------------------------
    # OHLC
    # ----------------------------------------

    result = pd.DataFrame(
        {
            "open": pd.to_numeric(
                df[lookup["open"]].to_numpy(),
                errors="coerce"
            ),

            "close": pd.to_numeric(
                df[lookup["close"]].to_numpy(),
                errors="coerce"
            )
        },
        index=idx
    )

    result = (
        result[
            ~result.index.isna()
        ]
        .dropna(
            subset=[
                "open",
                "close"
            ]
        )
        .sort_index()
    )

    if result.index.has_duplicates:

        result = result[
            ~result.index.duplicated(
                keep="last"
            )
        ]

    return result


def performance(returns):

    r = np.asarray(
        returns,
        dtype=float
    )

    wins = r[
        r > 0
    ].sum()

    losses = -r[
        r < 0
    ].sum()

    profit_factor = (
        np.inf
        if losses == 0
        else wins / losses
    )

    equity = np.cumprod(
        1.0 + r
    )

    peak = np.maximum.accumulate(
        equity
    )

    drawdown = (
        equity / peak
        -
        1.0
    )

    growth = float(
        equity[-1] - 1.0
    )

    max_dd = float(
        drawdown.min()
    )

    return_to_dd = (
        np.inf
        if max_dd == 0
        else growth / abs(max_dd)
    )

    return {
        "trades":
            int(len(r)),

        "win_rate":
            float(np.mean(r > 0)),

        "avg_return":
            float(np.mean(r)),

        "profit_factor":
            float(profit_factor),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            float(return_to_dd)
    }


# ============================================================
# MAIN REPLAY
# ============================================================

def main():

    banner(
        "FINAL 2026 PAPER EXECUTION REPLAY"
    )

    # ========================================================
    # 1. HISTORICAL REFERENCE
    # ========================================================

    source = globals().get(
        "HISTORICAL_REPLAY_LIVE_TRADES"
    )

    if not isinstance(
        source,
        pd.DataFrame
    ):

        return (
            False,
            "STOP_HISTORICAL_REPLAY_LIVE_TRADES_NOT_FOUND"
        )


    # ========================================================
    # 2. CANONICAL HISTORY
    # ========================================================

    canonical_raw = globals().get(
        "PRODUCTION_CANONICAL_HISTORY"
    )

    canonical = normalize_canonical(
        canonical_raw
    )

    if canonical is None:

        return (
            False,
            "STOP_PRODUCTION_CANONICAL_HISTORY_NOT_FOUND"
        )


    # ========================================================
    # 3. COLUMN RESOLUTION
    # ========================================================

    source = source.copy()

    signal_col = find_col(
        source,
        [
            "signal_time",
            "_diag_signal_time"
        ]
    )

    entry_col = find_col(
        source,
        [
            "entry_time"
        ]
    )

    exit_col = find_col(
        source,
        [
            "exit_time"
        ]
    )

    side_col = find_col(
        source,
        [
            "side"
        ]
    )

    size_col = find_col(
        source,
        [
            "position_size"
        ]
    )

    gross_col = find_col(
        source,
        [
            "gross_return"
        ]
    )

    net_col = find_col(
        source,
        [
            "net_return"
        ]
    )


    missing = []

    for name, col in {

        "signal_time":
            signal_col,

        "entry_time":
            entry_col,

        "exit_time":
            exit_col,

        "side":
            side_col,

        "position_size":
            size_col,

        "gross_return":
            gross_col,

        "net_return":
            net_col

    }.items():

        if col is None:
            missing.append(
                name
            )


    if missing:

        print(
            "Missing columns:",
            missing
        )

        return (
            False,
            "STOP_MISSING_REFERENCE_COLUMNS"
        )


    # ========================================================
    # 4. NORMALIZE REFERENCE
    # ========================================================

    source = source.reset_index(
        drop=True
    )


    ref = pd.DataFrame(
        {
            "trade_id":
                np.arange(
                    len(source),
                    dtype=int
                ),

            "signal_time":
                pd.to_datetime(
                    source[signal_col],
                    utc=True,
                    errors="coerce"
                ).to_numpy(),

            "entry_time":
                pd.to_datetime(
                    source[entry_col],
                    utc=True,
                    errors="coerce"
                ).to_numpy(),

            "exit_time":
                pd.to_datetime(
                    source[exit_col],
                    utc=True,
                    errors="coerce"
                ).to_numpy(),

            "side":
                source[
                    side_col
                ]
                .astype(str)
                .str.upper()
                .str.strip()
                .to_numpy(),

            "position_size":
                pd.to_numeric(
                    source[size_col],
                    errors="coerce"
                ).to_numpy(),

            "expected_gross_return":
                pd.to_numeric(
                    source[gross_col],
                    errors="coerce"
                ).to_numpy(),

            "expected_net_return":
                pd.to_numeric(
                    source[net_col],
                    errors="coerce"
                ).to_numpy()
        }
    )


    print(
        "Historical rows:",
        len(ref)
    )


    if len(ref) != EXPECTED_TRADES:

        print(
            "Expected:",
            EXPECTED_TRADES
        )

        print(
            "Found:",
            len(ref)
        )

        return (
            False,
            "STOP_REFERENCE_TRADE_COUNT_MISMATCH"
        )


    if ref.isna().any().any():

        print(
            ref
            .isna()
            .sum()[
                ref.isna().sum() > 0
            ]
        )

        return (
            False,
            "STOP_REFERENCE_HAS_NAN"
        )


    # ========================================================
    # 5. TIMING / INTEGRITY
    # ========================================================

    signal_entry_ok = bool(

        (
            ref["entry_time"]
            -
            ref["signal_time"]
        )

        .eq(
            pd.Timedelta(
                minutes=15
            )
        )

        .all()
    )


    entry_exit_ok = bool(

        (
            ref["exit_time"]
            -
            ref["entry_time"]
        )

        .eq(
            pd.Timedelta(
                minutes=30
            )
        )

        .all()
    )


    duplicate_signals = int(

        ref[
            "signal_time"
        ]
        .duplicated()
        .sum()
    )


    tmp = (

        ref
        .sort_values(
            [
                "entry_time",
                "trade_id"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    previous_exit = (

        tmp[
            "exit_time"
        ]
        .shift(1)
    )


    overlap_count = int(

        (
            tmp[
                "entry_time"
            ]
            <
            previous_exit
        )

        .fillna(False)

        .sum()
    )


    banner(
        "TIMING / INTEGRITY"
    )

    print(
        "Signal -> Entry = 15m:",
        signal_entry_ok
    )

    print(
        "Entry -> Exit = 30m:",
        entry_exit_ok
    )

    print(
        "Duplicate signals:",
        duplicate_signals
    )

    print(
        "Historical overlap:",
        overlap_count
    )


    # ========================================================
    # 6. COST CONTRACT
    # ========================================================

    expected_from_cost = (

        ref[
            "position_size"
        ]

        *

        (
            ref[
                "expected_gross_return"
            ]

            -

            BASE_COST
        )
    )


    max_cost_diff = float(

        np.abs(

            expected_from_cost

            -

            ref[
                "expected_net_return"
            ]

        ).max()
    )


    cost_ok = bool(
        max_cost_diff
        <=
        NET_TOL
    )


    banner(
        "FROZEN COST CONTRACT"
    )

    print(
        "net_return = "
        "position_size * "
        "(gross_return - 4e-05)"
    )

    print(
        "Max cost-contract diff:",
        max_cost_diff
    )

    print(
        "Passed:",
        cost_ok
    )


    if not (

        signal_entry_ok
        and
        entry_exit_ok
        and
        duplicate_signals == 0
        and
        overlap_count == 0
        and
        cost_ok

    ):

        return (
            False,
            "STOP_REFERENCE_CONTRACT_FAILED"
        )


    # ========================================================
    # 7. CANONICAL DATA
    # ========================================================

    banner(
        "CANONICAL HISTORY"
    )

    print(
        "Rows:",
        len(canonical)
    )

    print(
        "Period:",
        canonical.index.min(),
        "->",
        canonical.index.max()
    )


    # ========================================================
    # 8. EVENT STREAM
    # ========================================================

    events = []


    for row in ref.itertuples(
        index=False
    ):

        # EXIT priority = 0
        # ENTRY priority = 1
        #
        # therefore EXIT happens first
        # at the same timestamp.

        events.append(
            {
                "event_time":
                    row.entry_time,

                "priority":
                    1,

                "event":
                    "ENTRY",

                "trade_id":
                    row.trade_id
            }
        )


        events.append(
            {
                "event_time":
                    row.exit_time,

                "priority":
                    0,

                "event":
                    "EXIT",

                "trade_id":
                    row.trade_id
            }
        )


    events = (

        pd.DataFrame(
            events
        )

        .sort_values(
            [
                "event_time",
                "priority",
                "trade_id"
            ]
        )

        .reset_index(
            drop=True
        )
    )


    banner(
        "EVENT STREAM"
    )

    print(
        "Events:",
        len(events)
    )

    print(
        "ENTRY:",
        int(
            (
                events["event"]
                ==
                "ENTRY"
            ).sum()
        )
    )

    print(
        "EXIT:",
        int(
            (
                events["event"]
                ==
                "EXIT"
            ).sum()
        )
    )

    print(
        "Same-time priority:",
        "EXIT -> ENTRY"
    )


    # ========================================================
    # 9. PAPER ENGINE
    # ========================================================

    ref_by_id = ref.set_index(
        "trade_id"
    )

    open_position = None

    equity = float(
        STARTING_EQUITY
    )

    trades = []

    errors = []


    banner(
        "RUN PAPER ENGINE"
    )


    for ev in events.itertuples(
        index=False
    ):

        trade_id = int(
            ev.trade_id
        )

        event_time = pd.Timestamp(
            ev.event_time
        )

        expected = ref_by_id.loc[
            trade_id
        ]


        # ====================================================
        # EXIT
        # ====================================================

        if ev.event == "EXIT":

            if open_position is None:

                errors.append(
                    {
                        "time":
                            event_time,

                        "trade_id":
                            trade_id,

                        "error":
                            "EXIT_WITHOUT_POSITION"
                    }
                )

                continue


            if (

                open_position[
                    "trade_id"
                ]

                !=

                trade_id

            ):

                errors.append(
                    {
                        "time":
                            event_time,

                        "trade_id":
                            trade_id,

                        "error":
                            "WRONG_POSITION_AT_EXIT",

                        "open_trade_id":
                            open_position[
                                "trade_id"
                            ]
                    }
                )

                continue


            # Frozen exit:
            # PREVIOUS_BAR_CLOSE

            exit_price_time = (

                event_time

                -

                pd.Timedelta(
                    minutes=15
                )
            )


            if (

                exit_price_time

                not in

                canonical.index

            ):

                errors.append(
                    {
                        "time":
                            event_time,

                        "trade_id":
                            trade_id,

                        "error":
                            "MISSING_EXIT_PRICE"
                    }
                )

                continue


            exit_price = float(

                canonical.at[
                    exit_price_time,
                    "close"
                ]
            )


            entry_price = float(

                open_position[
                    "entry_price"
                ]
            )


            raw_return = (

                exit_price
                /
                entry_price
                -
                1.0
            )


            if (

                open_position[
                    "side"
                ]

                ==
                "BUY"

            ):

                gross_return = (
                    raw_return
                )

            else:

                gross_return = (
                    -raw_return
                )


            position_size = float(

                open_position[
                    "position_size"
                ]
            )


            # Frozen cost contract

            net_return = (

                position_size

                *

                (
                    gross_return

                    -

                    BASE_COST
                )
            )


            equity_before = (
                equity
            )


            equity *= (
                1.0
                +
                net_return
            )


            trades.append(
                {
                    "trade_id":
                        trade_id,

                    "signal_time":
                        open_position[
                            "signal_time"
                        ],

                    "entry_time":
                        open_position[
                            "entry_time"
                        ],

                    "exit_time":
                        event_time,

                    "side":
                        open_position[
                            "side"
                        ],

                    "position_size":
                        position_size,

                    "entry_price":
                        entry_price,

                    "exit_price":
                        exit_price,

                    "gross_return":
                        gross_return,

                    "net_return":
                        net_return,

                    "equity_before":
                        equity_before,

                    "equity_after":
                        equity,

                    "expected_gross_return":
                        float(
                            expected[
                                "expected_gross_return"
                            ]
                        ),

                    "expected_net_return":
                        float(
                            expected[
                                "expected_net_return"
                            ]
                        )
                }
            )


            open_position = None


        # ====================================================
        # ENTRY
        # ====================================================

        else:

            if open_position is not None:

                errors.append(
                    {
                        "time":
                            event_time,

                        "trade_id":
                            trade_id,

                        "error":
                            "OVERLAP_ENTRY",

                        "open_trade_id":
                            open_position[
                                "trade_id"
                            ]
                    }
                )

                continue


            # Frozen entry:
            # OPEN_AT_TIME

            if (

                event_time

                not in

                canonical.index

            ):

                errors.append(
                    {
                        "time":
                            event_time,

                        "trade_id":
                            trade_id,

                        "error":
                            "MISSING_ENTRY_PRICE"
                    }
                )

                continue


            open_position = {

                "trade_id":
                    trade_id,

                "signal_time":
                    expected[
                        "signal_time"
                    ],

                "entry_time":
                    event_time,

                "side":
                    expected[
                        "side"
                    ],

                "position_size":
                    float(
                        expected[
                            "position_size"
                        ]
                    ),

                "entry_price":
                    float(
                        canonical.at[
                            event_time,
                            "open"
                        ]
                    )
            }


    # ========================================================
    # 10. ENGINE RESULT
    # ========================================================

    replay = pd.DataFrame(
        trades
    )

    error_df = pd.DataFrame(
        errors
    )


    banner(
        "EXECUTION RESULT"
    )

    print(
        "Expected trades:",
        EXPECTED_TRADES
    )

    print(
        "Executed trades:",
        len(replay)
    )

    print(
        "Errors:",
        len(error_df)
    )

    print(
        "Final position flat:",
        open_position is None
    )


    if len(error_df) > 0:

        display(
            error_df.head(
                20
            )
        )


    if (

        len(replay)
        !=
        EXPECTED_TRADES

        or

        len(error_df)
        !=
        0

        or

        open_position
        is not None

    ):

        globals()[
            "FINAL_2026_PAPER_REPLAY"
        ] = replay

        globals()[
            "FINAL_2026_PAPER_REPLAY_ERRORS"
        ] = error_df

        return (
            False,
            "STOP_EXECUTION_STATE_MISMATCH"
        )


    # ========================================================
    # 11. TRADE PARITY
    # ========================================================

    replay[
        "gross_diff"
    ] = (

        replay[
            "gross_return"
        ]

        -

        replay[
            "expected_gross_return"
        ]
    )


    replay[
        "net_diff"
    ] = (

        replay[
            "net_return"
        ]

        -

        replay[
            "expected_net_return"
        ]
    )


    max_gross_diff = float(

        np.abs(

            replay[
                "gross_diff"
            ]

        ).max()
    )


    max_net_diff = float(

        np.abs(

            replay[
                "net_diff"
            ]

        ).max()
    )


    gross_ok = bool(
        max_gross_diff
        <=
        RET_TOL
    )


    net_ok = bool(
        max_net_diff
        <=
        NET_TOL
    )


    banner(
        "TRADE PARITY"
    )

    print(
        "Max gross diff:",
        max_gross_diff
    )

    print(
        "Gross parity:",
        gross_ok
    )

    print(
        "Max net diff:",
        max_net_diff
    )

    print(
        "Net parity:",
        net_ok
    )


    # ========================================================
    # 12. PERFORMANCE PARITY
    # ========================================================

    expected_perf = performance(

        ref[
            "expected_net_return"
        ]
    )


    replay_perf = performance(

        replay[
            "net_return"
        ]
    )


    perf_rows = []

    perf_ok = True


    for key in expected_perf:

        a = expected_perf[
            key
        ]

        b = replay_perf[
            key
        ]


        if (

            np.isinf(a)

            and

            np.isinf(b)

        ):

            diff = 0.0

            same = True

        else:

            diff = float(
                b - a
            )

            same = bool(

                np.isfinite(a)

                and

                np.isfinite(b)

                and

                abs(diff)
                <=
                RET_TOL
            )


        if key == "trades":

            same = (
                a == b
            )


        perf_ok = (
            perf_ok
            and
            same
        )


        perf_rows.append(
            {
                "metric":
                    key,

                "expected":
                    a,

                "replay":
                    b,

                "diff":
                    diff,

                "passed":
                    same
            }
        )


    perf_df = pd.DataFrame(
        perf_rows
    )


    banner(
        "PERFORMANCE PARITY"
    )

    display(
        perf_df
    )


    # ========================================================
    # 13. FINAL CHECKS
    # ========================================================

    checks = {

        "reference_300":
            len(ref) == 300,

        "signal_to_entry_15m":
            signal_entry_ok,

        "entry_to_exit_30m":
            entry_exit_ok,

        "duplicate_zero":
            duplicate_signals == 0,

        "overlap_zero":
            overlap_count == 0,

        "cost_contract":
            cost_ok,

        "executed_300":
            len(replay) == 300,

        "engine_errors_zero":
            len(error_df) == 0,

        "final_position_flat":
            open_position is None,

        "gross_parity":
            gross_ok,

        "net_parity":
            net_ok,

        "performance_parity":
            perf_ok
    }


    checks_df = pd.DataFrame(
        {
            "check":
                list(
                    checks.keys()
                ),

            "passed":
                list(
                    checks.values()
                )
        }
    )


    passed = bool(
        all(
            checks.values()
        )
    )


    if passed:

        decision = (
            "PASS_READY_FOR_FORWARD_PAPER_TRADING"
        )

    else:

        decision = (
            "STOP_FINAL_REPLAY_PARITY_FAILED"
        )


    globals()[
        "FINAL_2026_PAPER_REPLAY"
    ] = replay.copy()


    globals()[
        "FINAL_2026_PAPER_REPLAY_ERRORS"
    ] = error_df.copy()


    globals()[
        "FINAL_2026_PAPER_REPLAY_CHECKS"
    ] = checks_df.copy()


    globals()[
        "FINAL_2026_PAPER_REPLAY_PERFORMANCE"
    ] = perf_df.copy()


    banner(
        "FINAL CHECKS"
    )

    display(
        checks_df
    )


    print(
        "Starting equity:",
        f"{STARTING_EQUITY:,.2f}"
    )

    print(
        "Ending equity:",
        f"{equity:,.2f}"
    )


    return (
        passed,
        decision
    )


# ============================================================
# CONTROLLED RUN
# ============================================================

FINAL_2026_PAPER_REPLAY_PASSED = False

FINAL_2026_PAPER_REPLAY_DECISION = (
    "NOT_RUN"
)


try:

    (
        FINAL_2026_PAPER_REPLAY_PASSED,
        FINAL_2026_PAPER_REPLAY_DECISION

    ) = main()


except Exception as e:

    FINAL_2026_PAPER_REPLAY_PASSED = (
        False
    )

    FINAL_2026_PAPER_REPLAY_DECISION = (
        "STOP_UNEXPECTED_EXCEPTION"
    )


    banner(
        "CONTROLLED STOP"
    )


    print(
        "Error type:",
        type(e).__name__
    )

    print(
        "Message:",
        str(e)
    )

    print(
        "No real order was sent."
    )

    print(
        "Champion/model/features were not modified."
    )


# ============================================================
# SAVE FINAL STATUS
# ============================================================

globals()[
    "FINAL_2026_PAPER_REPLAY_PASSED"
] = FINAL_2026_PAPER_REPLAY_PASSED


globals()[
    "FINAL_2026_PAPER_REPLAY_DECISION"
] = FINAL_2026_PAPER_REPLAY_DECISION


# ============================================================
# CELL COMPLETE
# ============================================================

banner(
    "CELL COMPLETE"
)

print(
    "Passed:",
    FINAL_2026_PAPER_REPLAY_PASSED
)

print(
    "Decision:",
    FINAL_2026_PAPER_REPLAY_DECISION
)
